In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/cyrinemejrii/daataa/v7_document_vision_summary.csv
/kaggle/input/datasets/cyrinemejrii/daataa/v7_page_vision_predictions.csv
/kaggle/input/datasets/cyrinemejrii/daataa/c8_deep_ambiguity_predictions.csv
/kaggle/input/datasets/cyrinemejrii/daataa/c8_document_ambiguity_summary.csv
/kaggle/input/datasets/cyrinemejrii/daataa/v5_cv_model_comparison (2).csv
/kaggle/input/datasets/cyrinemejrii/daataa/v6_gradcam_xai_samples.csv


# MODULE C9 — MULTIMODAL REQUIREMENT QUALITY SCORING

This module combines NLP requirement intelligence with Computer Vision page intelligence.

Inputs:
- C8 deep ambiguity predictions from the NLP pipeline
- V7 page vision predictions from the Computer Vision pipeline
- V7 document-level vision summary

The goal is to build a multimodal quality scoring system that evaluates each requirement using:
- FR/NFR classification confidence
- NFR subtype confidence
- ambiguity prediction
- ambiguity probability
- section context
- page type
- visual confidence
- document visual quality

The output is a requirement-level, section-level, and document-level quality assessment.

In [2]:
# =========================================================
# MODULE C9.0 — CHECK MULTIMODAL INPUT FILES
# =========================================================

from pathlib import Path
import os
import re
import pandas as pd
import numpy as np

print("=" * 80)
print("MODULE C9.0 — CHECK MULTIMODAL INPUT FILES")
print("=" * 80)

def find_file_in_kaggle(filename, search_roots=["/kaggle/input", "/kaggle/working"]):
    matches = []
    
    for root in search_roots:
        root_path = Path(root)
        if root_path.exists():
            for path in root_path.rglob(filename):
                matches.append(path)
    
    matches = sorted(list(set(matches)))
    return matches


required_files = {
    "c8_requirements": "c8_deep_ambiguity_predictions.csv",
    "c8_document_summary": "c8_document_ambiguity_summary.csv",
    "v7_page_vision": "v7_page_vision_predictions.csv",
    "v7_document_vision": "v7_document_vision_summary.csv",
    "v5_cv_comparison": "v5_cv_model_comparison.csv",
    "v6_gradcam_xai": "v6_gradcam_xai_samples.csv"
}

found_paths = {}

for key, filename in required_files.items():
    matches = find_file_in_kaggle(filename)
    
    print(f"\n{key} — {filename}")
    if len(matches) == 0:
        print("  NOT FOUND")
        found_paths[key] = None
    else:
        for m in matches:
            print(" ", m)
        found_paths[key] = matches[0]

print("\nSelected paths:")
for key, path in found_paths.items():
    print(key, "=>", path)

# Critical files
critical_keys = [
    "c8_requirements",
    "v7_page_vision",
    "v7_document_vision"
]

for key in critical_keys:
    if found_paths[key] is None:
        raise FileNotFoundError(
            f"Missing critical file for C9: {required_files[key]}. "
            "Please add Notebook 2 and Notebook 3 outputs as Kaggle input datasets."
        )

MODULE C9.0 — CHECK MULTIMODAL INPUT FILES

c8_requirements — c8_deep_ambiguity_predictions.csv
  /kaggle/input/datasets/cyrinemejrii/daataa/c8_deep_ambiguity_predictions.csv

c8_document_summary — c8_document_ambiguity_summary.csv
  /kaggle/input/datasets/cyrinemejrii/daataa/c8_document_ambiguity_summary.csv

v7_page_vision — v7_page_vision_predictions.csv
  /kaggle/input/datasets/cyrinemejrii/daataa/v7_page_vision_predictions.csv

v7_document_vision — v7_document_vision_summary.csv
  /kaggle/input/datasets/cyrinemejrii/daataa/v7_document_vision_summary.csv

v5_cv_comparison — v5_cv_model_comparison.csv
  NOT FOUND

v6_gradcam_xai — v6_gradcam_xai_samples.csv
  /kaggle/input/datasets/cyrinemejrii/daataa/v6_gradcam_xai_samples.csv

Selected paths:
c8_requirements => /kaggle/input/datasets/cyrinemejrii/daataa/c8_deep_ambiguity_predictions.csv
c8_document_summary => /kaggle/input/datasets/cyrinemejrii/daataa/c8_document_ambiguity_summary.csv
v7_page_vision => /kaggle/input/datasets/cyrin

In [3]:
# =========================================================
# MODULE C9.1 — LOAD NLP AND VISION OUTPUTS
# =========================================================

df_c8 = pd.read_csv(found_paths["c8_requirements"])
df_v7_pages = pd.read_csv(found_paths["v7_page_vision"])
df_v7_docs = pd.read_csv(found_paths["v7_document_vision"])

print("C8 requirement-level NLP shape:", df_c8.shape)
print("V7 page-level vision shape:", df_v7_pages.shape)
print("V7 document-level vision shape:", df_v7_docs.shape)

print("\nC8 columns:")
print(df_c8.columns.tolist())

print("\nV7 page vision columns:")
print(df_v7_pages.columns.tolist())

print("\nV7 document vision columns:")
print(df_v7_docs.columns.tolist())

display(df_c8.head(5))
display(df_v7_pages.head(5))
display(df_v7_docs.head(5))

C8 requirement-level NLP shape: (3608, 25)
V7 page-level vision shape: (5666, 16)
V7 document-level vision shape: (124, 12)

C8 columns:
['doc_id', 'page_num', 'page_type', 'section_label', 'block_id', 'block_text', 'requirement_type_candidate', 'requirement_strength', 'requirement_confidence', 'deep_prediction', 'deep_confidence', 'deep_prediction_label', 'prediction_status', 'final_prediction', 'nfr_subtype_pred', 'nfr_subtype_confidence', 'nfr_subtype_model_name', 'ambiguity_pred_id', 'ambiguity_prediction', 'ambiguity_confidence', 'clear_probability', 'ambiguous_probability', 'ambiguity_model_name', 'ambiguity_status', 'final_ambiguity_label']

V7 page vision columns:
['doc_id', 'page_num', 'image_path', 'true_page_type', 'vision_model_name', 'vision_pred_id', 'vision_page_type', 'vision_confidence', 'vision_is_correct', 'vision_prob_appendix_page', 'vision_prob_content_page', 'vision_prob_cover_page', 'vision_prob_low_text_page', 'vision_prob_toc_page', 'vision_quality_flag', 'spl

,doc_id,page_num,page_type,section_label,block_id,block_text,requirement_type_candidate,requirement_strength,requirement_confidence,deep_prediction,...,nfr_subtype_confidence,nfr_subtype_model_name,ambiguity_pred_id,ambiguity_prediction,ambiguity_confidence,clear_probability,ambiguous_probability,ambiguity_model_name,ambiguity_status,final_ambiguity_label
0,0000 cctns,4,content_page,functional_requirements,9,"following investigation, police shall take the...",FR,strong,1.0,0,...,0.000000,roberta-base,0,CLEAR,0.979505,0.979505,0.020495,roberta-base,HIGH_CONFIDENCE,CLEAR
1,0000 cctns,6,content_page,non_functional_requirements,9,The solution should provide detailed context-s...,NFR,medium,0.9,1,...,0.884130,roberta-base,1,AMBIGUOUS,0.830338,0.169662,0.830338,roberta-base,HIGH_CONFIDENCE,AMBIGUOUS
2,0000 cctns,6,content_page,non_functional_requirements,12,The help should be accessible to the users bot...,NFR,medium,0.9,1,...,0.343726,roberta-base,1,AMBIGUOUS,0.988495,0.011505,0.988495,roberta-base,HIGH_CONFIDENCE,AMBIGUOUS
3,0000 cctns,6,content_page,non_functional_requirements,15,The solution should provide an interface for t...,NFR,medium,0.9,1,...,0.990577,roberta-base,0,CLEAR,0.957156,0.957156,0.042844,roberta-base,HIGH_CONFIDENCE,CLEAR
4,0000 cctns,6,content_page,non_functional_requirements,18,"The solution should send alerts (e.g., email, ...",NFR,medium,0.9,0,...,0.000000,roberta-base,0,CLEAR,0.948820,0.948820,0.051180,roberta-base,HIGH_CONFIDENCE,CLEAR


,doc_id,page_num,image_path,true_page_type,vision_model_name,vision_pred_id,vision_page_type,vision_confidence,vision_is_correct,vision_prob_appendix_page,vision_prob_content_page,vision_prob_cover_page,vision_prob_low_text_page,vision_prob_toc_page,vision_quality_flag,split
0,0000 cctns,1,/kaggle/input/datasets/cyrinemejrii/page-image...,cover_page,pretrained_resnet18,2,cover_page,0.998174,True,0.000098,0.000824,0.998174,0.000894,0.000011,high_confidence_visual_prediction,validation
1,0000 cctns,2,/kaggle/input/datasets/cyrinemejrii/page-image...,toc_page,pretrained_resnet18,1,content_page,0.822459,False,0.137331,0.822459,0.000021,0.000061,0.040128,medium_confidence_visual_prediction,validation
2,0000 cctns,3,/kaggle/input/datasets/cyrinemejrii/page-image...,content_page,pretrained_resnet18,1,content_page,0.758445,True,0.239660,0.758445,0.000125,0.000102,0.001667,medium_confidence_visual_prediction,validation
3,0000 cctns,4,/kaggle/input/datasets/cyrinemejrii/page-image...,content_page,pretrained_resnet18,0,appendix_page,0.617763,False,0.617763,0.373769,0.000001,0.000001,0.008465,medium_confidence_visual_prediction,validation
4,0000 cctns,5,/kaggle/input/datasets/cyrinemejrii/page-image...,content_page,pretrained_resnet18,1,content_page,0.944118,True,0.054365,0.944118,0.000007,0.000004,0.001506,high_confidence_visual_prediction,validation


,doc_id,total_pages,avg_vision_confidence,correct_visual_pages,cover_pages_pred,toc_pages_pred,content_pages_pred,appendix_pages_pred,low_text_pages_pred,low_confidence_pages,vision_accuracy_proxy,document_vision_quality_flag
0,0000 cctns,19,0.7431,14,1,0,16,2,0,4,0.737,moderate_visual_structure
1,0000 cctns scanned,19,0.7966,17,1,1,17,0,0,1,0.895,moderate_visual_structure
2,0000 gamma j,44,0.8798,37,1,1,20,12,10,3,0.841,moderate_visual_structure
3,0000 gamma j scanned,44,0.8651,28,1,1,20,11,11,3,0.636,moderate_visual_structure
4,0000 inventory,31,0.8988,29,1,2,27,1,0,1,0.935,moderate_visual_structure


In [4]:
# =========================================================
# MODULE C9.2 — REQUIRED COLUMNS CHECK
# =========================================================

required_c8_cols = [
    "doc_id",
    "page_num",
    "block_id",
    "block_text",
    "section_label",
    "page_type",
    "deep_prediction_label",
    "deep_confidence",
    "final_prediction",
    "final_ambiguity_label",
    "ambiguity_confidence",
    "ambiguous_probability"
]

required_v7_page_cols = [
    "doc_id",
    "page_num",
    "true_page_type",
    "vision_page_type",
    "vision_confidence",
    "vision_quality_flag"
]

required_v7_doc_cols = [
    "doc_id",
    "avg_vision_confidence",
    "document_vision_quality_flag",
    "low_confidence_pages",
    "vision_accuracy_proxy"
]

for col in required_c8_cols:
    if col not in df_c8.columns:
        raise ValueError(f"Missing required C8 column: {col}")

for col in required_v7_page_cols:
    if col not in df_v7_pages.columns:
        raise ValueError(f"Missing required V7 page column: {col}")

for col in required_v7_doc_cols:
    if col not in df_v7_docs.columns:
        raise ValueError(f"Missing required V7 document column: {col}")

print("All required columns are available.")

All required columns are available.


In [5]:
# =========================================================
# MODULE C9.3 — NORMALIZE MERGE KEYS
# =========================================================

def normalize_doc_id(x):
    return (
        str(x)
        .lower()
        .replace("-", " ")
        .replace("_", " ")
        .strip()
    )

def clean_spaces(x):
    return re.sub(r"\s+", " ", str(x)).strip()

for df in [df_c8, df_v7_pages, df_v7_docs]:
    df["doc_id"] = df["doc_id"].apply(normalize_doc_id).apply(clean_spaces)

df_c8["page_num"] = df_c8["page_num"].astype(int)
df_v7_pages["page_num"] = df_v7_pages["page_num"].astype(int)

print("C8 unique documents:", df_c8["doc_id"].nunique())
print("V7 page unique documents:", df_v7_pages["doc_id"].nunique())
print("V7 document unique documents:", df_v7_docs["doc_id"].nunique())

print("\nC8 page_num range:", df_c8["page_num"].min(), "→", df_c8["page_num"].max())
print("V7 page_num range:", df_v7_pages["page_num"].min(), "→", df_v7_pages["page_num"].max())

C8 unique documents: 24
V7 page unique documents: 124
V7 document unique documents: 124

C8 page_num range: 2 → 176
V7 page_num range: 1 → 176


In [6]:
# =========================================================
# MODULE C9.4 — MERGE REQUIREMENT NLP WITH PAGE-LEVEL VISION
# =========================================================

vision_page_cols = [
    "doc_id",
    "page_num",
    "true_page_type",
    "vision_page_type",
    "vision_confidence",
    "vision_quality_flag"
]

# Add probability columns if available
vision_prob_cols = [
    col for col in df_v7_pages.columns
    if col.startswith("vision_prob_")
]

vision_page_cols += vision_prob_cols

df_c9_multimodal = df_c8.merge(
    df_v7_pages[vision_page_cols],
    on=["doc_id", "page_num"],
    how="left"
)

print("C9 multimodal requirement-level shape:", df_c9_multimodal.shape)

print("\nMissing vision_page_type:", df_c9_multimodal["vision_page_type"].isna().sum())
print("Missing vision_confidence:", df_c9_multimodal["vision_confidence"].isna().sum())

print("\nVision page type distribution after merge:")
print(df_c9_multimodal["vision_page_type"].value_counts(dropna=False))

display(df_c9_multimodal.head(20))

C9 multimodal requirement-level shape: (3608, 34)

Missing vision_page_type: 0
Missing vision_confidence: 0

Vision page type distribution after merge:
vision_page_type
content_page     2883
appendix_page     658
toc_page           67
Name: count, dtype: int64


,doc_id,page_num,page_type,section_label,block_id,block_text,requirement_type_candidate,requirement_strength,requirement_confidence,deep_prediction,...,final_ambiguity_label,true_page_type,vision_page_type,vision_confidence,vision_quality_flag,vision_prob_appendix_page,vision_prob_content_page,vision_prob_cover_page,vision_prob_low_text_page,vision_prob_toc_page
0,0000 cctns,4,content_page,functional_requirements,9,"following investigation, police shall take the...",FR,strong,1.0,0,...,CLEAR,content_page,appendix_page,0.617763,medium_confidence_visual_prediction,0.617763,0.373769,0.000001,0.000001,0.008465
1,0000 cctns,6,content_page,non_functional_requirements,9,The solution should provide detailed context-s...,NFR,medium,0.9,1,...,AMBIGUOUS,content_page,content_page,0.664405,medium_confidence_visual_prediction,0.321971,0.664405,0.001336,0.008590,0.003698
2,0000 cctns,6,content_page,non_functional_requirements,12,The help should be accessible to the users bot...,NFR,medium,0.9,1,...,AMBIGUOUS,content_page,content_page,0.664405,medium_confidence_visual_prediction,0.321971,0.664405,0.001336,0.008590,0.003698
3,0000 cctns,6,content_page,non_functional_requirements,15,The solution should provide an interface for t...,NFR,medium,0.9,1,...,CLEAR,content_page,content_page,0.664405,medium_confidence_visual_prediction,0.321971,0.664405,0.001336,0.008590,0.003698
4,0000 cctns,6,content_page,non_functional_requirements,18,"The solution should send alerts (e.g., email, ...",NFR,medium,0.9,0,...,CLEAR,content_page,content_page,0.664405,medium_confidence_visual_prediction,0.321971,0.664405,0.001336,0.008590,0.003698
5,0000 cctns,6,content_page,non_functional_requirements,21,The solution should enable the user to track t...,NFR,medium,0.9,0,...,UNCERTAIN,content_page,content_page,0.664405,medium_confidence_visual_prediction,0.321971,0.664405,0.001336,0.008590,0.003698
6,0000 cctns,6,content_page,non_functional_requirements,24,The solution should enable the help-desk user ...,NFR,medium,0.9,0,...,CLEAR,content_page,content_page,0.664405,medium_confidence_visual_prediction,0.321971,0.664405,0.001336,0.008590,0.003698
7,0000 cctns,6,content_page,non_functional_requirements,28,The support solution should be accessible to t...,NFR,medium,0.9,1,...,AMBIGUOUS,content_page,content_page,0.664405,medium_confidence_visual_prediction,0.321971,0.664405,0.001336,0.008590,0.003698
8,0000 cctns,6,content_page,non_functional_requirements,35,System must keep an unalterable audit trail ca...,NFR,strong,1.0,1,...,CLEAR,content_page,content_page,0.664405,medium_confidence_visual_prediction,0.321971,0.664405,0.001336,0.008590,0.003698
9,0000 cctns,8,content_page,non_functional_requirements,5,The System must allow a user to be a member of...,NFR,strong,1.0,0,...,CLEAR,content_page,content_page,0.921359,high_confidence_visual_prediction,0.038727,0.921359,0.000001,0.000007,0.039905


In [7]:
# =========================================================
# MODULE C9.5 — MERGE DOCUMENT-LEVEL VISION SUMMARY
# =========================================================

vision_doc_cols = [
    "doc_id",
    "avg_vision_confidence",
    "document_vision_quality_flag",
    "low_confidence_pages",
    "vision_accuracy_proxy"
]

df_c9_multimodal = df_c9_multimodal.merge(
    df_v7_docs[vision_doc_cols],
    on="doc_id",
    how="left"
)

print("C9 multimodal shape after document-level vision merge:", df_c9_multimodal.shape)

print("\nMissing document vision flag:", df_c9_multimodal["document_vision_quality_flag"].isna().sum())

print("\nDocument vision quality distribution:")
print(df_c9_multimodal["document_vision_quality_flag"].value_counts(dropna=False))

display(df_c9_multimodal[
    [
        "doc_id",
        "page_num",
        "block_text",
        "section_label",
        "page_type",
        "deep_prediction_label",
        "final_prediction",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence",
        "vision_quality_flag",
        "document_vision_quality_flag"
    ]
].head(20))

C9 multimodal shape after document-level vision merge: (3608, 38)

Missing document vision flag: 0

Document vision quality distribution:
document_vision_quality_flag
moderate_visual_structure    3511
strong_visual_structure        97
Name: count, dtype: int64


,doc_id,page_num,block_text,section_label,page_type,deep_prediction_label,final_prediction,final_ambiguity_label,vision_page_type,vision_confidence,vision_quality_flag,document_vision_quality_flag
0,0000 cctns,4,"following investigation, police shall take the...",functional_requirements,content_page,FR,FR,CLEAR,appendix_page,0.617763,medium_confidence_visual_prediction,moderate_visual_structure
1,0000 cctns,6,The solution should provide detailed context-s...,non_functional_requirements,content_page,NFR,UNCERTAIN,AMBIGUOUS,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
2,0000 cctns,6,The help should be accessible to the users bot...,non_functional_requirements,content_page,NFR,NFR,AMBIGUOUS,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
3,0000 cctns,6,The solution should provide an interface for t...,non_functional_requirements,content_page,NFR,NFR,CLEAR,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
4,0000 cctns,6,"The solution should send alerts (e.g., email, ...",non_functional_requirements,content_page,FR,FR,CLEAR,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
5,0000 cctns,6,The solution should enable the user to track t...,non_functional_requirements,content_page,FR,FR,UNCERTAIN,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
6,0000 cctns,6,The solution should enable the help-desk user ...,non_functional_requirements,content_page,FR,FR,CLEAR,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
7,0000 cctns,6,The support solution should be accessible to t...,non_functional_requirements,content_page,NFR,NFR,AMBIGUOUS,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
8,0000 cctns,6,System must keep an unalterable audit trail ca...,non_functional_requirements,content_page,NFR,NFR,CLEAR,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
9,0000 cctns,8,The System must allow a user to be a member of...,non_functional_requirements,content_page,FR,UNCERTAIN,CLEAR,content_page,0.921359,high_confidence_visual_prediction,moderate_visual_structure


In [8]:
# =========================================================
# MODULE C9.6 — SAVE MULTIMODAL BASE TABLE
# =========================================================

c9_base_path = "/kaggle/working/c9_multimodal_base_table.csv"

df_c9_multimodal.to_csv(c9_base_path, index=False)

print("Saved:", c9_base_path)
print("C9 multimodal base table shape:", df_c9_multimodal.shape)

Saved: /kaggle/working/c9_multimodal_base_table.csv
C9 multimodal base table shape: (3608, 38)


In [9]:
# =========================================================
# MODULE C9.7 — BUILD MULTIMODAL QUALITY SIGNALS
# =========================================================

import re
import numpy as np
import pandas as pd

print("=" * 80)
print("MODULE C9.7 — BUILD MULTIMODAL QUALITY SIGNALS")
print("=" * 80)

df_c9_quality = df_c9_multimodal.copy()

df_c9_quality["block_text"] = df_c9_quality["block_text"].astype(str)


def safe_lower(x):
    return str(x).lower().strip() if pd.notna(x) else ""


def count_words(text):
    return len(str(text).split())


def has_mandatory_modal(text):
    text = safe_lower(text)
    return bool(re.search(r"\b(shall|must|required to|has to|will)\b", text))


def has_soft_modal(text):
    text = safe_lower(text)
    return bool(re.search(r"\b(should|may|could|might|preferably|where possible)\b", text))


def has_numeric_constraint(text):
    text = safe_lower(text)
    return bool(
        re.search(
            r"\b\d+(\.\d+)?\s*(seconds?|minutes?|hours?|days?|ms|milliseconds?|%|percent|users?|transactions?|requests?|records?|mb|gb|kb|times?)\b",
            text
        )
    )


def has_threshold_language(text):
    text = safe_lower(text)
    return bool(
        re.search(
            r"\b(within|at least|at most|no more than|no less than|maximum|minimum|less than|greater than|equal to|between|not exceed|under|over)\b",
            text
        )
    )


def has_vague_terms(text):
    text = safe_lower(text)

    vague_terms = [
        "quickly", "fast", "soon", "easy", "easily", "efficient", "efficiently",
        "user friendly", "simple", "robust", "secure", "reliable", "appropriate",
        "adequate", "sufficient", "as needed", "as required", "etc", "where possible",
        "normally", "generally", "reasonable", "minimal", "optimal", "flexible",
        "seamless", "intuitive", "acceptable", "high quality", "low latency"
    ]

    return any(re.search(r"\b" + re.escape(term) + r"\b", text) for term in vague_terms)


def has_actor_or_system(text):
    text = safe_lower(text)
    return bool(re.search(r"\b(system|user|administrator|admin|operator|application|service|module|interface)\b", text))


def get_float(row, col, default=0.0):
    try:
        value = row.get(col, default)
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def build_quality_signals(row):
    text = str(row["block_text"])
    word_count = count_words(text)

    final_pred = str(row.get("final_prediction", ""))
    deep_pred = str(row.get("deep_prediction_label", ""))
    deep_conf = get_float(row, "deep_confidence", 0.0)

    final_ambiguity = str(row.get("final_ambiguity_label", ""))
    ambiguity_conf = get_float(row, "ambiguity_confidence", 0.0)
    ambiguous_prob = get_float(row, "ambiguous_probability", 0.0)

    nfr_subtype_conf = get_float(row, "nfr_subtype_confidence", 0.0)

    requirement_confidence = get_float(row, "requirement_confidence", 0.0)

    section_label = row.get("section_label", None)

    vision_page_type = str(row.get("vision_page_type", "unknown"))
    true_page_type = str(row.get("true_page_type", "unknown"))
    vision_confidence = get_float(row, "vision_confidence", 0.0)
    vision_quality_flag = str(row.get("vision_quality_flag", "unknown"))
    document_vision_quality_flag = str(row.get("document_vision_quality_flag", "unknown"))
    avg_vision_confidence = get_float(row, "avg_vision_confidence", 0.0)

    return pd.Series({
        "quality_word_count": word_count,

        "has_mandatory_modal": has_mandatory_modal(text),
        "has_soft_modal": has_soft_modal(text),
        "has_numeric_constraint": has_numeric_constraint(text),
        "has_threshold_language": has_threshold_language(text),
        "has_vague_terms": has_vague_terms(text),
        "has_actor_or_system": has_actor_or_system(text),

        "is_fr": deep_pred == "FR",
        "is_nfr": deep_pred == "NFR",
        "is_prediction_uncertain": final_pred == "UNCERTAIN",
        "is_deep_low_confidence": deep_conf < 0.75,

        "is_ambiguous": final_ambiguity == "AMBIGUOUS",
        "is_ambiguity_uncertain": final_ambiguity == "UNCERTAIN",
        "ambiguous_probability_signal": ambiguous_prob,
        "ambiguity_confidence_signal": ambiguity_conf,

        "is_nfr_subtype_low_confidence": (
            deep_pred == "NFR" and nfr_subtype_conf < 0.50
        ),

        "missing_section_context": (
            pd.isna(section_label)
            or str(section_label).strip().lower() in ["", "nan", "none", "unknown", "unknown_section"]
        ),

        "low_requirement_confidence": requirement_confidence < 0.70,

        # Computer Vision signals
        "vision_page_mismatch": vision_page_type != true_page_type,
        "low_vision_confidence": vision_confidence < 0.60,
        "medium_vision_confidence": 0.60 <= vision_confidence < 0.85,
        "high_vision_confidence": vision_confidence >= 0.85,

        "vision_says_non_content_page": vision_page_type in [
            "cover_page",
            "toc_page",
            "low_text_page"
        ],

        "true_page_is_content_or_appendix": true_page_type in [
            "content_page",
            "appendix_page"
        ],

        "weak_document_visual_structure": document_vision_quality_flag == "weak_visual_structure",
        "moderate_document_visual_structure": document_vision_quality_flag == "moderate_visual_structure",
        "strong_document_visual_structure": document_vision_quality_flag == "strong_visual_structure",

        "avg_vision_confidence_signal": avg_vision_confidence,
        "vision_confidence_signal": vision_confidence
    })


df_quality_signals = df_c9_quality.apply(build_quality_signals, axis=1)

df_c9_quality = pd.concat(
    [df_c9_quality.reset_index(drop=True), df_quality_signals.reset_index(drop=True)],
    axis=1
)

print("C9 quality table shape:", df_c9_quality.shape)

display(df_c9_quality[
    [
        "doc_id",
        "page_num",
        "block_text",
        "final_prediction",
        "deep_confidence",
        "final_ambiguity_label",
        "ambiguous_probability",
        "vision_page_type",
        "vision_confidence",
        "vision_quality_flag",
        "document_vision_quality_flag",
        "quality_word_count",
        "has_vague_terms",
        "has_numeric_constraint",
        "is_ambiguous",
        "low_vision_confidence",
        "vision_page_mismatch"
    ]
].head(25))

MODULE C9.7 — BUILD MULTIMODAL QUALITY SIGNALS
C9 quality table shape: (3608, 67)


,doc_id,page_num,block_text,final_prediction,deep_confidence,final_ambiguity_label,ambiguous_probability,vision_page_type,vision_confidence,vision_quality_flag,document_vision_quality_flag,quality_word_count,has_vague_terms,has_numeric_constraint,is_ambiguous,low_vision_confidence,vision_page_mismatch
0,0000 cctns,4,"following investigation, police shall take the...",FR,0.992321,CLEAR,0.020495,appendix_page,0.617763,medium_confidence_visual_prediction,moderate_visual_structure,11,False,False,False,False,True
1,0000 cctns,6,The solution should provide detailed context-s...,UNCERTAIN,0.509114,AMBIGUOUS,0.830338,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure,11,False,False,True,False,False
2,0000 cctns,6,The help should be accessible to the users bot...,NFR,0.776161,AMBIGUOUS,0.988495,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure,15,False,False,True,False,False
3,0000 cctns,6,The solution should provide an interface for t...,NFR,0.903017,CLEAR,0.042844,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure,14,False,False,False,False,False
4,0000 cctns,6,"The solution should send alerts (e.g., email, ...",FR,0.957549,CLEAR,0.051180,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure,15,False,False,False,False,False
5,0000 cctns,6,The solution should enable the user to track t...,FR,0.942391,UNCERTAIN,0.280699,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure,12,False,False,False,False,False
6,0000 cctns,6,The solution should enable the help-desk user ...,FR,0.913040,CLEAR,0.217484,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure,13,False,False,False,False,False
7,0000 cctns,6,The support solution should be accessible to t...,NFR,0.794371,AMBIGUOUS,0.989949,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure,13,False,False,True,False,False
8,0000 cctns,6,System must keep an unalterable audit trail ca...,NFR,0.950417,CLEAR,0.019154,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure,11,False,False,False,False,False
9,0000 cctns,8,The System must allow a user to be a member of...,UNCERTAIN,0.583352,CLEAR,0.013407,content_page,0.921359,high_confidence_visual_prediction,moderate_visual_structure,15,False,False,False,False,False


In [10]:
# =========================================================
# MODULE C9.8 — CREATE SILVER QUALITY LABELS
# =========================================================

print("=" * 80)
print("MODULE C9.8 — CREATE SILVER QUALITY LABELS")
print("=" * 80)


def compute_silver_quality_score(row):
    score = 100.0

    word_count = int(row["quality_word_count"])

    # NLP classification reliability
    if row["is_prediction_uncertain"]:
        score -= 15

    if row["is_deep_low_confidence"]:
        score -= 10

    if row["low_requirement_confidence"]:
        score -= 5

    # Ambiguity
    if row["is_ambiguous"]:
        score -= 28

    if row["is_ambiguity_uncertain"]:
        score -= 12

    score -= float(row["ambiguous_probability_signal"]) * 12

    # NFR subtype reliability
    if row["is_nfr_subtype_low_confidence"]:
        score -= 8

    # Text quality
    if word_count < 6:
        score -= 18

    if word_count > 80:
        score -= 8

    if not row["has_mandatory_modal"] and not row["has_soft_modal"]:
        score -= 10

    if not row["has_actor_or_system"]:
        score -= 6

    if row["has_vague_terms"]:
        score -= 12

    # NFR measurability
    if row["is_nfr"]:
        if not row["has_numeric_constraint"] and not row["has_threshold_language"]:
            score -= 14

    # Section context
    if row["missing_section_context"]:
        score -= 5

    # Computer Vision penalties
    if row["low_vision_confidence"]:
        score -= 8

    if row["vision_page_mismatch"]:
        score -= 6

    if row["vision_says_non_content_page"]:
        score -= 10

    if row["weak_document_visual_structure"]:
        score -= 8
    elif row["moderate_document_visual_structure"]:
        score -= 3

    return max(0, min(100, round(score, 2)))


def silver_quality_label(score):
    if score >= 75:
        return "HIGH_QUALITY"
    elif score >= 50:
        return "MEDIUM_QUALITY"
    else:
        return "LOW_QUALITY"


df_c9_quality["silver_quality_score"] = df_c9_quality.apply(
    compute_silver_quality_score,
    axis=1
)

df_c9_quality["silver_quality_label"] = df_c9_quality["silver_quality_score"].apply(
    silver_quality_label
)

label2id_quality = {
    "LOW_QUALITY": 0,
    "MEDIUM_QUALITY": 1,
    "HIGH_QUALITY": 2
}

id2label_quality = {
    0: "LOW_QUALITY",
    1: "MEDIUM_QUALITY",
    2: "HIGH_QUALITY"
}

df_c9_quality["quality_label_id"] = df_c9_quality["silver_quality_label"].map(label2id_quality)

print("Silver quality label distribution:")
print(df_c9_quality["silver_quality_label"].value_counts())

print("\nSilver quality score summary:")
display(df_c9_quality["silver_quality_score"].describe())

display(df_c9_quality[
    [
        "doc_id",
        "page_num",
        "block_text",
        "silver_quality_score",
        "silver_quality_label",
        "final_prediction",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence",
        "vision_quality_flag",
        "document_vision_quality_flag"
    ]
].head(30))

MODULE C9.8 — CREATE SILVER QUALITY LABELS
Silver quality label distribution:
silver_quality_label
HIGH_QUALITY      2053
MEDIUM_QUALITY     948
LOW_QUALITY        607
Name: count, dtype: int64

Silver quality score summary:


count    3608.000000
mean       71.820937
std        20.210379
min         0.000000
25%        58.507500
50%        79.195000
75%        85.830000
max        99.810000
Name: silver_quality_score, dtype: float64

,doc_id,page_num,block_text,silver_quality_score,silver_quality_label,final_prediction,final_ambiguity_label,vision_page_type,vision_confidence,vision_quality_flag,document_vision_quality_flag
0,0000 cctns,4,"following investigation, police shall take the...",90.75,HIGH_QUALITY,FR,CLEAR,appendix_page,0.617763,medium_confidence_visual_prediction,moderate_visual_structure
1,0000 cctns,6,The solution should provide detailed context-s...,14.04,LOW_QUALITY,UNCERTAIN,AMBIGUOUS,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
2,0000 cctns,6,The help should be accessible to the users bot...,29.14,LOW_QUALITY,NFR,AMBIGUOUS,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
3,0000 cctns,6,The solution should provide an interface for t...,82.49,HIGH_QUALITY,NFR,CLEAR,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
4,0000 cctns,6,"The solution should send alerts (e.g., email, ...",96.39,HIGH_QUALITY,FR,CLEAR,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
5,0000 cctns,6,The solution should enable the user to track t...,81.63,HIGH_QUALITY,FR,UNCERTAIN,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
6,0000 cctns,6,The solution should enable the help-desk user ...,94.39,HIGH_QUALITY,FR,CLEAR,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
7,0000 cctns,6,The support solution should be accessible to t...,43.12,LOW_QUALITY,NFR,AMBIGUOUS,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
8,0000 cctns,6,System must keep an unalterable audit trail ca...,82.77,HIGH_QUALITY,NFR,CLEAR,content_page,0.664405,medium_confidence_visual_prediction,moderate_visual_structure
9,0000 cctns,8,The System must allow a user to be a member of...,71.84,MEDIUM_QUALITY,UNCERTAIN,CLEAR,content_page,0.921359,high_confidence_visual_prediction,moderate_visual_structure


In [11]:
# =========================================================
# MODULE C9.9 — BALANCE QUALITY TRAINING DATA
# =========================================================

print("=" * 80)
print("MODULE C9.9 — BALANCE QUALITY TRAINING DATA")
print("=" * 80)

MAX_PER_CLASS = 1800

df_quality_train = (
    df_c9_quality
    .dropna(subset=["block_text", "quality_label_id"])
    .copy()
)

df_quality_train["block_text"] = df_quality_train["block_text"].astype(str)

df_quality_train_balanced = (
    df_quality_train
    .groupby("quality_label_id", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(len(x), MAX_PER_CLASS),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("Original quality training shape:", df_quality_train.shape)
print("Balanced quality training shape:", df_quality_train_balanced.shape)

print("\nOriginal label distribution:")
print(df_quality_train["silver_quality_label"].value_counts())

print("\nBalanced label distribution:")
print(df_quality_train_balanced["silver_quality_label"].value_counts())

display(df_quality_train_balanced[
    [
        "block_text",
        "silver_quality_score",
        "silver_quality_label",
        "quality_label_id",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence"
    ]
].head(20))

MODULE C9.9 — BALANCE QUALITY TRAINING DATA
Original quality training shape: (3608, 70)
Balanced quality training shape: (3355, 70)

Original label distribution:
silver_quality_label
HIGH_QUALITY      2053
MEDIUM_QUALITY     948
LOW_QUALITY        607
Name: count, dtype: int64

Balanced label distribution:
silver_quality_label
HIGH_QUALITY      1800
MEDIUM_QUALITY     948
LOW_QUALITY        607
Name: count, dtype: int64


/tmp/ipykernel_1260/1826616129.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,block_text,silver_quality_score,silver_quality_label,quality_label_id,final_ambiguity_label,vision_page_type,vision_confidence
0,"error (i.e., it should be on).",46.61,LOW_QUALITY,0,AMBIGUOUS,content_page,0.991808
1,"To meet growth requirements, the TCS should be...",13.71,LOW_QUALITY,0,AMBIGUOUS,content_page,0.565475
2,3.1.1 Description And Priority Customers will ...,49.21,LOW_QUALITY,0,UNCERTAIN,content_page,0.811857
3,Pre: conditions: Administrator must be able to...,44.82,LOW_QUALITY,0,CLEAR,content_page,0.592811
4,cooling unit should be requested.,28.25,LOW_QUALITY,0,AMBIGUOUS,content_page,0.988898
5,This input to the macro indicates to the SCE-M...,28.84,LOW_QUALITY,0,AMBIGUOUS,content_page,0.506005
6,windows or pop-up windows should only be opene...,24.24,LOW_QUALITY,0,AMBIGUOUS,content_page,0.592405
7,The capacity will be deﬁned in Section 2.9 on ...,43.59,LOW_QUALITY,0,CLEAR,content_page,0.993474
8,readout registers will be clocked in the same ...,20.52,LOW_QUALITY,0,UNCERTAIN,content_page,0.508693
9,system are available in this mode. Under no ci...,32.91,LOW_QUALITY,0,AMBIGUOUS,content_page,0.998137


In [12]:
# =========================================================
# MODULE C9.10 — TRAIN / VALIDATION SPLIT
# =========================================================

from sklearn.model_selection import train_test_split
import pandas as pd

print("=" * 80)
print("MODULE C9.10 — TRAIN / VALIDATION SPLIT")
print("=" * 80)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_quality_train_balanced["block_text"].astype(str).tolist(),
    df_quality_train_balanced["quality_label_id"].astype(int).tolist(),
    test_size=0.15,
    stratify=df_quality_train_balanced["quality_label_id"],
    random_state=42
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

print("\nTrain label distribution:")
print(pd.Series(train_labels).map(id2label_quality).value_counts())

print("\nValidation label distribution:")
print(pd.Series(val_labels).map(id2label_quality).value_counts())

MODULE C9.10 — TRAIN / VALIDATION SPLIT
Train size: 2851
Validation size: 504

Train label distribution:
HIGH_QUALITY      1530
MEDIUM_QUALITY     805
LOW_QUALITY        516
Name: count, dtype: int64

Validation label distribution:
HIGH_QUALITY      270
MEDIUM_QUALITY    143
LOW_QUALITY        91
Name: count, dtype: int64


In [13]:
# =========================================================
# MODULE C9.11 — TOKENIZATION AND TORCH DATASET WITH ROBERTA
# =========================================================

import torch
from transformers import AutoTokenizer

print("=" * 80)
print("MODULE C9.11 — TOKENIZATION AND TORCH DATASET WITH ROBERTA")
print("=" * 80)

QUALITY_MODEL_NAME = "roberta-base"

quality_tokenizer = AutoTokenizer.from_pretrained(QUALITY_MODEL_NAME)

train_encodings = quality_tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = quality_tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)


class RequirementQualityDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


train_quality_dataset = RequirementQualityDataset(train_encodings, train_labels)
val_quality_dataset = RequirementQualityDataset(val_encodings, val_labels)

print("Tokenizer:", QUALITY_MODEL_NAME)
print("Train dataset size:", len(train_quality_dataset))
print("Validation dataset size:", len(val_quality_dataset))

MODULE C9.11 — TOKENIZATION AND TORCH DATASET WITH ROBERTA


Tokenizer: roberta-base
Train dataset size: 2851
Validation dataset size: 504


In [14]:
# =========================================================
# MODULE C9.12 — TRAIN ROBERTA QUALITY CLASSIFIER
# =========================================================

import gc
import torch
import numpy as np

from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score

print("=" * 80)
print("MODULE C9.12 — TRAIN ROBERTA QUALITY CLASSIFIER")
print("=" * 80)

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

quality_model = AutoModelForSequenceClassification.from_pretrained(
    QUALITY_MODEL_NAME,
    num_labels=3,
    id2label=id2label_quality,
    label2id=label2id_quality
)

def compute_quality_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    precision, recall, _, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "precision": precision,
        "recall": recall
    }


training_args_quality = TrainingArguments(
    output_dir="/kaggle/working/roberta_requirement_quality",
    num_train_epochs=3,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 4,
    per_device_eval_batch_size=16 if torch.cuda.is_available() else 4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

quality_trainer = Trainer(
    model=quality_model,
    args=training_args_quality,
    train_dataset=train_quality_dataset,
    eval_dataset=val_quality_dataset,
    compute_metrics=compute_quality_metrics
)

quality_trainer.train()

quality_eval_results = quality_trainer.evaluate()

print("=" * 80)
print("ROBERTA QUALITY EVALUATION RESULTS")
print("=" * 80)
print(quality_eval_results)

quality_model_path = "/kaggle/working/roberta_requirement_quality_model"

quality_trainer.save_model(quality_model_path)
quality_tokenizer.save_pretrained(quality_model_path)

print("Saved RoBERTa quality model:", quality_model_path)

MODULE C9.12 — TRAIN ROBERTA QUALITY CLASSIFIER
Device: cpu


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument i

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision,Recall
1,0.875730,0.930192,0.642857,0.535873,0.606179,0.617933,0.642857
2,0.727777,0.743200,0.704365,0.622424,0.684697,0.696032,0.704365
3,0.567783,0.875525,0.686508,0.628685,0.680770,0.686865,0.686508


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

ROBERTA QUALITY EVALUATION RESULTS
{'eval_loss': 0.8755245804786682, 'eval_accuracy': 0.6865079365079365, 'eval_macro_f1': 0.6286847271299568, 'eval_weighted_f1': 0.6807697664853142, 'eval_precision': 0.686864551349536, 'eval_recall': 0.6865079365079365, 'eval_runtime': 116.5972, 'eval_samples_per_second': 4.323, 'eval_steps_per_second': 1.081, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved RoBERTa quality model: /kaggle/working/roberta_requirement_quality_model


In [15]:
# =========================================================
# MODULE C9.13 — APPLY DEEP QUALITY MODEL ON ALL REQUIREMENTS
# =========================================================

from tqdm.auto import tqdm
import torch
import numpy as np

print("=" * 80)
print("MODULE C9.13 — APPLY DEEP QUALITY MODEL ON ALL REQUIREMENTS")
print("=" * 80)

df_c9_deep_quality = df_c9_quality.copy()
df_c9_deep_quality = df_c9_deep_quality.dropna(subset=["block_text"]).copy()
df_c9_deep_quality["block_text"] = df_c9_deep_quality["block_text"].astype(str)

texts = df_c9_deep_quality["block_text"].tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
quality_model.to(device)
quality_model.eval()

BATCH_SIZE = 32 if torch.cuda.is_available() else 8

all_quality_preds = []
all_quality_confidences = []
all_low_quality_probs = []
all_medium_quality_probs = []
all_high_quality_probs = []

torch.cuda.empty_cache()

with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Predicting deep quality"):
        batch_texts = texts[i:i + BATCH_SIZE]

        batch_encodings = quality_tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt"
        )

        batch_encodings = {
            key: value.to(device)
            for key, value in batch_encodings.items()
        }

        outputs = quality_model(**batch_encodings)

        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_quality_preds.extend(preds.cpu().numpy())
        all_quality_confidences.extend(probs.max(dim=1).values.cpu().numpy())

        all_low_quality_probs.extend(probs[:, 0].cpu().numpy())
        all_medium_quality_probs.extend(probs[:, 1].cpu().numpy())
        all_high_quality_probs.extend(probs[:, 2].cpu().numpy())

        del batch_encodings, outputs, probs, preds
        torch.cuda.empty_cache()

df_c9_deep_quality["deep_quality_pred_id"] = all_quality_preds
df_c9_deep_quality["deep_quality_label"] = df_c9_deep_quality["deep_quality_pred_id"].map(id2label_quality)
df_c9_deep_quality["deep_quality_confidence"] = all_quality_confidences

df_c9_deep_quality["low_quality_probability"] = all_low_quality_probs
df_c9_deep_quality["medium_quality_probability"] = all_medium_quality_probs
df_c9_deep_quality["high_quality_probability"] = all_high_quality_probs

df_c9_deep_quality["quality_model_name"] = QUALITY_MODEL_NAME

print("Deep quality prediction distribution:")
print(df_c9_deep_quality["deep_quality_label"].value_counts())

print("\nDeep quality confidence summary:")
display(df_c9_deep_quality["deep_quality_confidence"].describe())

display(df_c9_deep_quality[
    [
        "doc_id",
        "page_num",
        "block_text",
        "silver_quality_score",
        "silver_quality_label",
        "deep_quality_label",
        "deep_quality_confidence",
        "low_quality_probability",
        "medium_quality_probability",
        "high_quality_probability",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence"
    ]
].head(30))

MODULE C9.13 — APPLY DEEP QUALITY MODEL ON ALL REQUIREMENTS


Predicting deep quality:   0%|          | 0/451 [00:00<?, ?it/s]

Deep quality prediction distribution:
deep_quality_label
HIGH_QUALITY      2058
MEDIUM_QUALITY    1048
LOW_QUALITY        502
Name: count, dtype: int64

Deep quality confidence summary:


count    3608.000000
mean        0.872127
std         0.138726
min         0.380512
25%         0.789724
50%         0.927515
75%         0.984425
max         0.989187
Name: deep_quality_confidence, dtype: float64

,doc_id,page_num,block_text,silver_quality_score,silver_quality_label,deep_quality_label,deep_quality_confidence,low_quality_probability,medium_quality_probability,high_quality_probability,final_ambiguity_label,vision_page_type,vision_confidence
0,0000 cctns,4,"following investigation, police shall take the...",90.75,HIGH_QUALITY,HIGH_QUALITY,0.984422,0.003155,0.012423,0.984422,CLEAR,appendix_page,0.617763
1,0000 cctns,6,The solution should provide detailed context-s...,14.04,LOW_QUALITY,LOW_QUALITY,0.878588,0.878588,0.114603,0.006809,AMBIGUOUS,content_page,0.664405
2,0000 cctns,6,The help should be accessible to the users bot...,29.14,LOW_QUALITY,LOW_QUALITY,0.922981,0.922981,0.070338,0.006681,AMBIGUOUS,content_page,0.664405
3,0000 cctns,6,The solution should provide an interface for t...,82.49,HIGH_QUALITY,HIGH_QUALITY,0.881594,0.018335,0.100071,0.881594,CLEAR,content_page,0.664405
4,0000 cctns,6,"The solution should send alerts (e.g., email, ...",96.39,HIGH_QUALITY,HIGH_QUALITY,0.966510,0.005895,0.027595,0.966510,CLEAR,content_page,0.664405
5,0000 cctns,6,The solution should enable the user to track t...,81.63,HIGH_QUALITY,HIGH_QUALITY,0.905181,0.015832,0.078987,0.905181,UNCERTAIN,content_page,0.664405
6,0000 cctns,6,The solution should enable the help-desk user ...,94.39,HIGH_QUALITY,MEDIUM_QUALITY,0.810194,0.100940,0.810194,0.088866,CLEAR,content_page,0.664405
7,0000 cctns,6,The support solution should be accessible to t...,43.12,LOW_QUALITY,LOW_QUALITY,0.921446,0.921446,0.072008,0.006547,AMBIGUOUS,content_page,0.664405
8,0000 cctns,6,System must keep an unalterable audit trail ca...,82.77,HIGH_QUALITY,MEDIUM_QUALITY,0.898860,0.054261,0.898860,0.046878,CLEAR,content_page,0.664405
9,0000 cctns,8,The System must allow a user to be a member of...,71.84,MEDIUM_QUALITY,HIGH_QUALITY,0.971259,0.004557,0.024184,0.971259,CLEAR,content_page,0.921359


In [16]:
# =========================================================
# MODULE C9.14 — MULTIDIMENSIONAL MULTIMODAL QUALITY SCORING
# =========================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("MODULE C9.14 — MULTIDIMENSIONAL MULTIMODAL QUALITY SCORING")
print("=" * 80)

df_c9_final = df_c9_deep_quality.copy()


def clamp_score(x):
    return max(0, min(100, round(float(x), 2)))


def as_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in ["true", "1", "yes"]


def compute_quality_dimensions(row):
    word_count = int(row.get("quality_word_count", 0))

    is_ambiguous = as_bool(row.get("is_ambiguous", False))
    is_ambiguity_uncertain = as_bool(row.get("is_ambiguity_uncertain", False))
    is_prediction_uncertain = as_bool(row.get("is_prediction_uncertain", False))
    is_deep_low_confidence = as_bool(row.get("is_deep_low_confidence", False))
    is_nfr = as_bool(row.get("is_nfr", False))
    is_nfr_subtype_low_confidence = as_bool(row.get("is_nfr_subtype_low_confidence", False))
    has_vague_terms = as_bool(row.get("has_vague_terms", False))
    has_mandatory_modal = as_bool(row.get("has_mandatory_modal", False))
    has_soft_modal = as_bool(row.get("has_soft_modal", False))
    has_numeric_constraint = as_bool(row.get("has_numeric_constraint", False))
    has_threshold_language = as_bool(row.get("has_threshold_language", False))
    has_actor_or_system = as_bool(row.get("has_actor_or_system", False))
    missing_section_context = as_bool(row.get("missing_section_context", False))

    low_vision_confidence = as_bool(row.get("low_vision_confidence", False))
    medium_vision_confidence = as_bool(row.get("medium_vision_confidence", False))
    vision_page_mismatch = as_bool(row.get("vision_page_mismatch", False))
    vision_says_non_content_page = as_bool(row.get("vision_says_non_content_page", False))
    weak_document_visual_structure = as_bool(row.get("weak_document_visual_structure", False))
    moderate_document_visual_structure = as_bool(row.get("moderate_document_visual_structure", False))

    deep_confidence = float(row.get("deep_confidence", 0.0))
    ambiguous_probability = float(row.get("ambiguous_probability_signal", 0.0))
    vision_confidence = float(row.get("vision_confidence_signal", 0.0))
    avg_vision_confidence = float(row.get("avg_vision_confidence_signal", 0.0))

    # 1. Clarity
    clarity = 100
    if has_vague_terms:
        clarity -= 25
    if is_ambiguous:
        clarity -= 35
    if is_ambiguity_uncertain:
        clarity -= 15
    if word_count < 6:
        clarity -= 20
    if word_count > 80:
        clarity -= 10

    # 2. Testability
    testability = 100
    if not has_mandatory_modal and not has_soft_modal:
        testability -= 25
    if has_vague_terms:
        testability -= 20
    if is_ambiguous:
        testability -= 30
    if not has_actor_or_system:
        testability -= 10

    # 3. Measurability
    measurability = 100
    if is_nfr:
        if not has_numeric_constraint:
            measurability -= 35
        if not has_threshold_language:
            measurability -= 25
    else:
        if has_vague_terms:
            measurability -= 10

    # 4. Classification reliability
    classification_reliability = 100
    classification_reliability -= (1 - deep_confidence) * 45
    if is_prediction_uncertain:
        classification_reliability -= 30
    if is_deep_low_confidence:
        classification_reliability -= 15
    if is_nfr_subtype_low_confidence:
        classification_reliability -= 12

    # 5. Ambiguity safety
    ambiguity_safety = 100
    ambiguity_safety -= ambiguous_probability * 70
    if is_ambiguous:
        ambiguity_safety -= 20
    if is_ambiguity_uncertain:
        ambiguity_safety -= 10

    # 6. Visual context
    visual_context = 100
    visual_context -= (1 - vision_confidence) * 30
    visual_context -= (1 - avg_vision_confidence) * 15

    if low_vision_confidence:
        visual_context -= 20
    elif medium_vision_confidence:
        visual_context -= 8

    if vision_page_mismatch:
        visual_context -= 15

    if vision_says_non_content_page:
        visual_context -= 18

    if weak_document_visual_structure:
        visual_context -= 20
    elif moderate_document_visual_structure:
        visual_context -= 8

    # 7. Completeness
    completeness = 100
    if missing_section_context:
        completeness -= 20
    if word_count < 6:
        completeness -= 25
    if not has_mandatory_modal and not has_soft_modal:
        completeness -= 15
    if not has_actor_or_system:
        completeness -= 12

    return pd.Series({
        "clarity_score": clamp_score(clarity),
        "testability_score": clamp_score(testability),
        "measurability_score": clamp_score(measurability),
        "classification_reliability_score": clamp_score(classification_reliability),
        "ambiguity_safety_score": clamp_score(ambiguity_safety),
        "visual_context_score": clamp_score(visual_context),
        "completeness_score": clamp_score(completeness)
    })


df_quality_dimensions = df_c9_final.apply(compute_quality_dimensions, axis=1)

df_c9_final = pd.concat(
    [df_c9_final.reset_index(drop=True), df_quality_dimensions.reset_index(drop=True)],
    axis=1
)

# Deep model numeric score
df_c9_final["deep_quality_numeric_score"] = (
    df_c9_final["low_quality_probability"] * 25 +
    df_c9_final["medium_quality_probability"] * 60 +
    df_c9_final["high_quality_probability"] * 90
).round(2)

# Final multimodal neuro-symbolic score
df_c9_final["requirement_quality_score"] = (
    0.18 * df_c9_final["clarity_score"] +
    0.16 * df_c9_final["testability_score"] +
    0.16 * df_c9_final["measurability_score"] +
    0.14 * df_c9_final["classification_reliability_score"] +
    0.14 * df_c9_final["ambiguity_safety_score"] +
    0.10 * df_c9_final["visual_context_score"] +
    0.07 * df_c9_final["completeness_score"] +
    0.05 * df_c9_final["deep_quality_numeric_score"]
).round(2)


def quality_level(score):
    if score >= 85:
        return "EXCELLENT"
    elif score >= 70:
        return "GOOD"
    elif score >= 55:
        return "REVIEW_NEEDED"
    elif score >= 40:
        return "WEAK"
    else:
        return "CRITICAL"


df_c9_final["requirement_quality_level"] = df_c9_final["requirement_quality_score"].apply(quality_level)

print("Requirement quality level distribution:")
print(df_c9_final["requirement_quality_level"].value_counts())

print("\nRequirement quality score summary:")
display(df_c9_final["requirement_quality_score"].describe())

display(df_c9_final[
    [
        "doc_id",
        "page_num",
        "block_text",
        "deep_quality_label",
        "deep_quality_confidence",
        "clarity_score",
        "testability_score",
        "measurability_score",
        "classification_reliability_score",
        "ambiguity_safety_score",
        "visual_context_score",
        "completeness_score",
        "requirement_quality_score",
        "requirement_quality_level"
    ]
].head(30))

MODULE C9.14 — MULTIDIMENSIONAL MULTIMODAL QUALITY SCORING
Requirement quality level distribution:
requirement_quality_level
EXCELLENT        2293
GOOD              913
REVIEW_NEEDED     344
WEAK               57
CRITICAL            1
Name: count, dtype: int64

Requirement quality score summary:


count    3608.000000
mean       86.281788
std        11.291704
min        39.440000
25%        80.000000
50%        91.395000
75%        95.260000
max        99.010000
Name: requirement_quality_score, dtype: float64

,doc_id,page_num,block_text,deep_quality_label,deep_quality_confidence,clarity_score,testability_score,measurability_score,classification_reliability_score,ambiguity_safety_score,visual_context_score,completeness_score,requirement_quality_score,requirement_quality_level
0,0000 cctns,4,"following investigation, police shall take the...",HIGH_QUALITY,0.984422,100.0,100.0,100.0,99.65,98.57,53.68,100.0,94.59,EXCELLENT
1,0000 cctns,6,The solution should provide detailed context-s...,LOW_QUALITY,0.878588,65.0,60.0,40.0,32.91,21.88,70.08,88.0,50.01,WEAK
2,0000 cctns,6,The help should be accessible to the users bot...,LOW_QUALITY,0.922981,65.0,60.0,40.0,77.93,10.81,70.08,88.0,54.69,WEAK
3,0000 cctns,6,The solution should provide an interface for t...,HIGH_QUALITY,0.881594,100.0,100.0,40.0,95.64,97.00,70.08,100.0,85.67,EXCELLENT
4,0000 cctns,6,"The solution should send alerts (e.g., email, ...",HIGH_QUALITY,0.966510,100.0,100.0,100.0,98.09,96.42,70.08,100.0,95.68,EXCELLENT
5,0000 cctns,6,The solution should enable the user to track t...,HIGH_QUALITY,0.905181,85.0,100.0,100.0,97.41,70.35,70.08,100.0,89.12,EXCELLENT
6,0000 cctns,6,The solution should enable the help-desk user ...,MEDIUM_QUALITY,0.810194,100.0,100.0,100.0,96.09,84.78,70.08,100.0,92.29,EXCELLENT
7,0000 cctns,6,The support solution should be accessible to t...,LOW_QUALITY,0.921446,65.0,60.0,65.0,78.75,10.70,70.08,88.0,58.79,REVIEW_NEEDED
8,0000 cctns,6,System must keep an unalterable audit trail ca...,MEDIUM_QUALITY,0.898860,100.0,100.0,40.0,97.77,98.66,70.08,100.0,84.88,GOOD
9,0000 cctns,8,The System must allow a user to be a member of...,HIGH_QUALITY,0.971259,100.0,100.0,100.0,36.25,99.06,85.79,100.0,88.97,EXCELLENT


In [17]:
# =========================================================
# MODULE C9.15 — QUALITY ISSUE DETECTION AND RECOMMENDATIONS
# =========================================================

print("=" * 80)
print("MODULE C9.15 — QUALITY ISSUE DETECTION AND RECOMMENDATIONS")
print("=" * 80)


def detect_quality_issues(row):
    issues = []

    if as_bool(row.get("is_ambiguous", False)):
        issues.append("ambiguous_requirement")

    if as_bool(row.get("is_ambiguity_uncertain", False)):
        issues.append("ambiguity_uncertain_review")

    if as_bool(row.get("is_prediction_uncertain", False)):
        issues.append("fr_nfr_classification_uncertain")

    if as_bool(row.get("is_deep_low_confidence", False)):
        issues.append("low_fr_nfr_confidence")

    if as_bool(row.get("is_nfr_subtype_low_confidence", False)):
        issues.append("low_nfr_subtype_confidence")

    if as_bool(row.get("has_vague_terms", False)):
        issues.append("vague_language")

    if as_bool(row.get("is_nfr", False)) and not as_bool(row.get("has_numeric_constraint", False)):
        issues.append("nfr_missing_numeric_target")

    if as_bool(row.get("is_nfr", False)) and not as_bool(row.get("has_threshold_language", False)):
        issues.append("nfr_missing_threshold")

    if not as_bool(row.get("has_mandatory_modal", False)) and not as_bool(row.get("has_soft_modal", False)):
        issues.append("missing_requirement_modal")

    if not as_bool(row.get("has_actor_or_system", False)):
        issues.append("missing_actor_or_system_context")

    if int(row.get("quality_word_count", 0)) < 6:
        issues.append("too_short_requirement")

    if int(row.get("quality_word_count", 0)) > 80:
        issues.append("too_long_requirement")

    if as_bool(row.get("missing_section_context", False)):
        issues.append("missing_section_context")

    if as_bool(row.get("low_vision_confidence", False)):
        issues.append("low_visual_confidence")

    if as_bool(row.get("vision_page_mismatch", False)):
        issues.append("vision_page_type_mismatch")

    if as_bool(row.get("vision_says_non_content_page", False)):
        issues.append("requirement_on_non_content_visual_page")

    if as_bool(row.get("weak_document_visual_structure", False)):
        issues.append("weak_document_visual_structure")

    if row.get("deep_quality_label", "") == "LOW_QUALITY":
        issues.append("deep_model_low_quality_risk")

    return sorted(list(set(issues)))


def generate_quality_recommendations(issues):
    recs = []

    if "ambiguous_requirement" in issues:
        recs.append("Rewrite the requirement to remove ambiguity and make the expected behavior explicit.")

    if "vague_language" in issues:
        recs.append("Replace vague terms with precise measurable wording.")

    if "nfr_missing_numeric_target" in issues:
        recs.append("Add a measurable numeric target such as response time, availability percentage, throughput, capacity, or error rate.")

    if "nfr_missing_threshold" in issues:
        recs.append("Add a threshold expression such as within, at least, at most, maximum, or minimum.")

    if "missing_requirement_modal" in issues:
        recs.append("Use a clear requirement modal such as shall or must.")

    if "missing_actor_or_system_context" in issues:
        recs.append("Specify the responsible actor or system component.")

    if "fr_nfr_classification_uncertain" in issues or "low_fr_nfr_confidence" in issues:
        recs.append("Review whether this requirement is functional or non-functional.")

    if "low_nfr_subtype_confidence" in issues:
        recs.append("Review the NFR subtype because the subtype classifier is uncertain.")

    if "too_short_requirement" in issues:
        recs.append("Add enough context to describe condition, actor, system behavior, and expected outcome.")

    if "too_long_requirement" in issues:
        recs.append("Split this requirement into smaller atomic requirements.")

    if "missing_section_context" in issues:
        recs.append("Verify the section assignment because the requirement lacks reliable document context.")

    if "low_visual_confidence" in issues:
        recs.append("Inspect the source page image because the vision model has low confidence.")

    if "vision_page_type_mismatch" in issues:
        recs.append("Review page type consistency between NLP labels and visual model prediction.")

    if "requirement_on_non_content_visual_page" in issues:
        recs.append("Check whether this text was extracted from a cover, table-of-contents, or low-text page.")

    if "weak_document_visual_structure" in issues:
        recs.append("Review document formatting and page structure because visual quality is weak.")

    if len(recs) == 0:
        recs.append("No major quality issue detected.")

    return recs


df_c9_final["quality_issues"] = df_c9_final.apply(detect_quality_issues, axis=1)
df_c9_final["quality_recommendations"] = df_c9_final["quality_issues"].apply(generate_quality_recommendations)
df_c9_final["num_quality_issues"] = df_c9_final["quality_issues"].apply(len)

print("Quality issues count summary:")
display(df_c9_final["num_quality_issues"].describe())

display(df_c9_final[
    [
        "doc_id",
        "page_num",
        "block_text",
        "requirement_quality_score",
        "requirement_quality_level",
        "quality_issues",
        "quality_recommendations"
    ]
].head(30))

MODULE C9.15 — QUALITY ISSUE DETECTION AND RECOMMENDATIONS
Quality issues count summary:


count    3608.000000
mean        2.237528
std         1.850935
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        10.000000
Name: num_quality_issues, dtype: float64

,doc_id,page_num,block_text,requirement_quality_score,requirement_quality_level,quality_issues,quality_recommendations
0,0000 cctns,4,"following investigation, police shall take the...",94.59,EXCELLENT,[vision_page_type_mismatch],[Review page type consistency between NLP labe...
1,0000 cctns,6,The solution should provide detailed context-s...,50.01,WEAK,"[ambiguous_requirement, deep_model_low_quality...",[Rewrite the requirement to remove ambiguity a...
2,0000 cctns,6,The help should be accessible to the users bot...,54.69,WEAK,"[ambiguous_requirement, deep_model_low_quality...",[Rewrite the requirement to remove ambiguity a...
3,0000 cctns,6,The solution should provide an interface for t...,85.67,EXCELLENT,"[nfr_missing_numeric_target, nfr_missing_thres...",[Add a measurable numeric target such as respo...
4,0000 cctns,6,"The solution should send alerts (e.g., email, ...",95.68,EXCELLENT,[],[No major quality issue detected.]
5,0000 cctns,6,The solution should enable the user to track t...,89.12,EXCELLENT,[ambiguity_uncertain_review],[No major quality issue detected.]
6,0000 cctns,6,The solution should enable the help-desk user ...,92.29,EXCELLENT,[],[No major quality issue detected.]
7,0000 cctns,6,The support solution should be accessible to t...,58.79,REVIEW_NEEDED,"[ambiguous_requirement, deep_model_low_quality...",[Rewrite the requirement to remove ambiguity a...
8,0000 cctns,6,System must keep an unalterable audit trail ca...,84.88,GOOD,"[nfr_missing_numeric_target, nfr_missing_thres...",[Add a measurable numeric target such as respo...
9,0000 cctns,8,The System must allow a user to be a member of...,88.97,EXCELLENT,"[fr_nfr_classification_uncertain, low_fr_nfr_c...",[Review whether this requirement is functional...


In [18]:
# =========================================================
# MODULE C9.16 — SECTION-LEVEL QUALITY SUMMARY
# =========================================================

print("=" * 80)
print("MODULE C9.16 — SECTION-LEVEL QUALITY SUMMARY")
print("=" * 80)

df_c9_final["section_label_clean"] = (
    df_c9_final["section_label"]
    .fillna("unknown_section")
    .astype(str)
    .replace({"nan": "unknown_section", "None": "unknown_section"})
)

df_section_quality_summary = (
    df_c9_final
    .groupby(["doc_id", "section_label_clean"])
    .agg(
        total_requirements=("block_text", "count"),
        avg_requirement_quality_score=("requirement_quality_score", "mean"),
        avg_clarity_score=("clarity_score", "mean"),
        avg_testability_score=("testability_score", "mean"),
        avg_measurability_score=("measurability_score", "mean"),
        avg_classification_reliability_score=("classification_reliability_score", "mean"),
        avg_ambiguity_safety_score=("ambiguity_safety_score", "mean"),
        avg_visual_context_score=("visual_context_score", "mean"),
        avg_completeness_score=("completeness_score", "mean"),
        critical_requirements=("requirement_quality_level", lambda x: (x == "CRITICAL").sum()),
        weak_requirements=("requirement_quality_level", lambda x: (x == "WEAK").sum()),
        review_needed_requirements=("requirement_quality_level", lambda x: (x == "REVIEW_NEEDED").sum()),
        good_requirements=("requirement_quality_level", lambda x: (x == "GOOD").sum()),
        excellent_requirements=("requirement_quality_level", lambda x: (x == "EXCELLENT").sum()),
        ambiguous_requirements=("final_ambiguity_label", lambda x: (x == "AMBIGUOUS").sum()),
        uncertain_ambiguity_requirements=("final_ambiguity_label", lambda x: (x == "UNCERTAIN").sum()),
        uncertain_fr_nfr_requirements=("final_prediction", lambda x: (x == "UNCERTAIN").sum()),
        low_vision_confidence_requirements=("low_vision_confidence", lambda x: sum(as_bool(v) for v in x)),
        vision_page_mismatch_requirements=("vision_page_mismatch", lambda x: sum(as_bool(v) for v in x))
    )
    .reset_index()
)

score_cols = [
    "avg_requirement_quality_score",
    "avg_clarity_score",
    "avg_testability_score",
    "avg_measurability_score",
    "avg_classification_reliability_score",
    "avg_ambiguity_safety_score",
    "avg_visual_context_score",
    "avg_completeness_score"
]

for col in score_cols:
    df_section_quality_summary[col] = df_section_quality_summary[col].round(2)

df_section_quality_summary["section_quality_level"] = (
    df_section_quality_summary["avg_requirement_quality_score"].apply(quality_level)
)

df_section_quality_summary["weak_or_critical_ratio"] = (
    (
        df_section_quality_summary["weak_requirements"] +
        df_section_quality_summary["critical_requirements"]
    ) /
    df_section_quality_summary["total_requirements"].replace(0, np.nan)
).fillna(0).round(3)

print("Section quality summary shape:", df_section_quality_summary.shape)
print("\nSection quality level distribution:")
print(df_section_quality_summary["section_quality_level"].value_counts())

display(df_section_quality_summary.head(50))

MODULE C9.16 — SECTION-LEVEL QUALITY SUMMARY
Section quality summary shape: (58, 23)

Section quality level distribution:
section_quality_level
EXCELLENT        31
GOOD             24
REVIEW_NEEDED     3
Name: count, dtype: int64


,doc_id,section_label_clean,total_requirements,avg_requirement_quality_score,avg_clarity_score,avg_testability_score,avg_measurability_score,avg_classification_reliability_score,avg_ambiguity_safety_score,avg_visual_context_score,...,review_needed_requirements,good_requirements,excellent_requirements,ambiguous_requirements,uncertain_ambiguity_requirements,uncertain_fr_nfr_requirements,low_vision_confidence_requirements,vision_page_mismatch_requirements,section_quality_level,weak_or_critical_ratio
0,0000 cctns,functional_requirements,1,94.59,100.00,100.00,100.00,99.65,98.57,53.68,...,0,0,1,0,0,0,0,1,EXCELLENT,0.000
1,0000 cctns,non_functional_requirements,108,71.21,77.31,77.04,72.45,78.36,47.68,67.55,...,43,21,29,59,9,29,48,0,GOOD,0.139
2,0000 cctns scanned,non_functional_requirements,33,74.82,74.24,80.00,76.82,78.59,55.60,79.52,...,8,11,10,15,3,9,0,0,GOOD,0.121
3,0000 gamma j,appendices,57,91.23,95.96,93.33,98.77,87.74,89.40,77.28,...,2,10,45,5,2,8,3,18,EXCELLENT,0.000
4,0000 gamma j,functional_requirements,20,92.79,100.00,92.50,91.00,89.32,97.55,87.64,...,0,3,17,0,0,3,0,0,EXCELLENT,0.000
5,0000 gamma j,interface_requirements,7,93.18,100.00,98.57,91.43,81.35,97.69,88.91,...,0,1,6,0,0,2,0,0,EXCELLENT,0.000
6,0000 gamma j,performance_requirements,39,81.74,91.67,92.82,56.92,80.83,86.96,75.16,...,4,16,19,5,0,8,0,0,GOOD,0.000
7,0000 gamma j scanned,appendices,43,90.22,91.86,92.33,92.67,94.05,89.09,74.83,...,3,5,34,3,3,2,6,12,EXCELLENT,0.023
8,0000 gamma j scanned,interface_requirements,13,88.03,94.62,96.15,72.31,91.49,90.40,78.79,...,1,1,11,1,0,1,0,0,EXCELLENT,0.000
9,0000 inventory,functional_requirements,3,93.75,100.00,93.33,100.00,99.85,95.20,70.98,...,0,0,3,0,0,0,0,0,EXCELLENT,0.000


In [19]:
# =========================================================
# MODULE C9.17 — DOCUMENT-LEVEL QUALITY SUMMARY
# =========================================================

print("=" * 80)
print("MODULE C9.17 — DOCUMENT-LEVEL QUALITY SUMMARY")
print("=" * 80)

df_document_quality_summary = (
    df_c9_final
    .groupby("doc_id")
    .agg(
        total_requirements=("block_text", "count"),
        document_quality_score=("requirement_quality_score", "mean"),
        avg_clarity_score=("clarity_score", "mean"),
        avg_testability_score=("testability_score", "mean"),
        avg_measurability_score=("measurability_score", "mean"),
        avg_classification_reliability_score=("classification_reliability_score", "mean"),
        avg_ambiguity_safety_score=("ambiguity_safety_score", "mean"),
        avg_visual_context_score=("visual_context_score", "mean"),
        avg_completeness_score=("completeness_score", "mean"),
        avg_deep_quality_confidence=("deep_quality_confidence", "mean"),
        avg_vision_confidence=("vision_confidence", "mean"),

        critical_requirements=("requirement_quality_level", lambda x: (x == "CRITICAL").sum()),
        weak_requirements=("requirement_quality_level", lambda x: (x == "WEAK").sum()),
        review_needed_requirements=("requirement_quality_level", lambda x: (x == "REVIEW_NEEDED").sum()),
        good_requirements=("requirement_quality_level", lambda x: (x == "GOOD").sum()),
        excellent_requirements=("requirement_quality_level", lambda x: (x == "EXCELLENT").sum()),

        ambiguous_requirements=("final_ambiguity_label", lambda x: (x == "AMBIGUOUS").sum()),
        uncertain_ambiguity_requirements=("final_ambiguity_label", lambda x: (x == "UNCERTAIN").sum()),
        uncertain_fr_nfr_requirements=("final_prediction", lambda x: (x == "UNCERTAIN").sum()),

        low_vision_confidence_requirements=("low_vision_confidence", lambda x: sum(as_bool(v) for v in x)),
        vision_page_mismatch_requirements=("vision_page_mismatch", lambda x: sum(as_bool(v) for v in x)),

        document_vision_quality_flag=("document_vision_quality_flag", "first")
    )
    .reset_index()
)

score_cols_doc = [
    "document_quality_score",
    "avg_clarity_score",
    "avg_testability_score",
    "avg_measurability_score",
    "avg_classification_reliability_score",
    "avg_ambiguity_safety_score",
    "avg_visual_context_score",
    "avg_completeness_score",
    "avg_deep_quality_confidence",
    "avg_vision_confidence"
]

for col in score_cols_doc:
    df_document_quality_summary[col] = df_document_quality_summary[col].round(2)

df_document_quality_summary["document_quality_level"] = (
    df_document_quality_summary["document_quality_score"].apply(quality_level)
)

df_document_quality_summary["critical_ratio"] = (
    df_document_quality_summary["critical_requirements"] /
    df_document_quality_summary["total_requirements"].replace(0, np.nan)
).fillna(0).round(3)

df_document_quality_summary["weak_or_critical_ratio"] = (
    (
        df_document_quality_summary["weak_requirements"] +
        df_document_quality_summary["critical_requirements"]
    ) /
    df_document_quality_summary["total_requirements"].replace(0, np.nan)
).fillna(0).round(3)

df_document_quality_summary["ambiguous_ratio"] = (
    df_document_quality_summary["ambiguous_requirements"] /
    df_document_quality_summary["total_requirements"].replace(0, np.nan)
).fillna(0).round(3)

df_document_quality_summary["vision_mismatch_ratio"] = (
    df_document_quality_summary["vision_page_mismatch_requirements"] /
    df_document_quality_summary["total_requirements"].replace(0, np.nan)
).fillna(0).round(3)


def document_review_priority(row):
    if (
        row["document_quality_score"] < 55
        or row["weak_or_critical_ratio"] >= 0.35
        or row["ambiguous_ratio"] >= 0.25
    ):
        return "HIGH_PRIORITY_REVIEW"

    if (
        row["document_quality_score"] < 70
        or row["weak_or_critical_ratio"] >= 0.15
        or row["ambiguous_ratio"] >= 0.10
    ):
        return "MEDIUM_PRIORITY_REVIEW"

    return "LOW_PRIORITY_REVIEW"


df_document_quality_summary["document_review_priority"] = (
    df_document_quality_summary.apply(document_review_priority, axis=1)
)

print("Document quality summary shape:", df_document_quality_summary.shape)

print("\nDocument quality level distribution:")
print(df_document_quality_summary["document_quality_level"].value_counts())

print("\nDocument review priority distribution:")
print(df_document_quality_summary["document_review_priority"].value_counts())

display(df_document_quality_summary.head(30))

MODULE C9.17 — DOCUMENT-LEVEL QUALITY SUMMARY
Document quality summary shape: (24, 29)

Document quality level distribution:
document_quality_level
EXCELLENT    14
GOOD         10
Name: count, dtype: int64

Document review priority distribution:
document_review_priority
MEDIUM_PRIORITY_REVIEW    10
LOW_PRIORITY_REVIEW        9
HIGH_PRIORITY_REVIEW       5
Name: count, dtype: int64


,doc_id,total_requirements,document_quality_score,avg_clarity_score,avg_testability_score,avg_measurability_score,avg_classification_reliability_score,avg_ambiguity_safety_score,avg_visual_context_score,avg_completeness_score,...,uncertain_fr_nfr_requirements,low_vision_confidence_requirements,vision_page_mismatch_requirements,document_vision_quality_flag,document_quality_level,critical_ratio,weak_or_critical_ratio,ambiguous_ratio,vision_mismatch_ratio,document_review_priority
0,0000 cctns,109,71.42,77.52,77.25,72.71,78.55,48.14,67.43,94.39,...,29,48,1,moderate_visual_structure,GOOD,0.009,0.138,0.541,0.009,HIGH_PRIORITY_REVIEW
1,0000 cctns scanned,33,74.82,74.24,80.00,76.82,78.59,55.60,79.52,98.18,...,9,0,0,moderate_visual_structure,GOOD,0.000,0.121,0.455,0.000,HIGH_PRIORITY_REVIEW
2,0000 gamma j,123,88.58,95.49,93.33,83.82,85.44,90.43,78.96,96.29,...,21,3,18,moderate_visual_structure,EXCELLENT,0.000,0.000,0.081,0.146,LOW_PRIORITY_REVIEW
3,0000 gamma j scanned,56,89.71,92.50,93.21,87.95,93.46,89.39,75.75,97.43,...,3,6,12,moderate_visual_structure,EXCELLENT,0.000,0.018,0.071,0.214,LOW_PRIORITY_REVIEW
4,0000 inventory,10,75.39,86.00,84.00,64.00,93.02,60.41,56.72,95.20,...,0,1,5,moderate_visual_structure,GOOD,0.000,0.100,0.400,0.500,HIGH_PRIORITY_REVIEW
5,0000 inventory scanned,7,74.34,79.29,82.86,82.86,80.99,52.02,69.13,78.29,...,2,0,3,strong_visual_structure,GOOD,0.000,0.000,0.429,0.429,HIGH_PRIORITY_REVIEW
6,1995 gemini,424,82.51,88.24,83.86,83.87,78.75,72.23,91.23,90.44,...,120,2,0,moderate_visual_structure,GOOD,0.000,0.031,0.241,0.000,MEDIUM_PRIORITY_REVIEW
7,1998 themas,117,89.49,91.79,88.12,98.72,94.53,79.44,85.63,91.78,...,9,7,0,moderate_visual_structure,EXCELLENT,0.000,0.000,0.162,0.000,MEDIUM_PRIORITY_REVIEW
8,1999 tcs,1107,88.86,97.84,91.85,88.50,83.46,95.47,71.41,91.78,...,223,230,202,moderate_visual_structure,EXCELLENT,0.000,0.005,0.020,0.182,LOW_PRIORITY_REVIEW
9,2001 beyond,100,78.71,83.20,81.70,92.40,85.62,61.18,60.66,91.55,...,19,38,29,moderate_visual_structure,GOOD,0.000,0.040,0.340,0.290,HIGH_PRIORITY_REVIEW


In [20]:
# =========================================================
# MODULE C9.18 — SAVE FINAL C9 OUTPUTS
# =========================================================

print("=" * 80)
print("MODULE C9.18 — SAVE FINAL C9 OUTPUTS")
print("=" * 80)

c9_requirement_output_path = "/kaggle/working/c9_multimodal_requirement_quality_scores.csv"
c9_section_summary_path = "/kaggle/working/c9_section_quality_summary.csv"
c9_document_summary_path = "/kaggle/working/c9_document_quality_summary.csv"
c9_eval_results_path = "/kaggle/working/c9_roberta_quality_eval_results.csv"
c9_manifest_path = "/kaggle/working/c9_outputs_manifest.csv"
c9_summary_path = "/kaggle/working/c9_final_summary.txt"

df_c9_final.to_csv(c9_requirement_output_path, index=False)
df_section_quality_summary.to_csv(c9_section_summary_path, index=False)
df_document_quality_summary.to_csv(c9_document_summary_path, index=False)

# Save eval results if available
try:
    pd.DataFrame([quality_eval_results]).to_csv(c9_eval_results_path, index=False)
except Exception:
    pd.DataFrame([{"note": "quality_eval_results variable not available"}]).to_csv(c9_eval_results_path, index=False)

c9_outputs = {
    "requirement_level_quality_scores": c9_requirement_output_path,
    "section_quality_summary": c9_section_summary_path,
    "document_quality_summary": c9_document_summary_path,
    "roberta_quality_eval_results": c9_eval_results_path,
    "quality_model_path": "/kaggle/working/roberta_requirement_quality_model"
}

df_c9_manifest = pd.DataFrame(
    [{"output_name": k, "path": v} for k, v in c9_outputs.items()]
)

df_c9_manifest.to_csv(c9_manifest_path, index=False)

summary_text = f"""
C9 — Multimodal Requirement Quality Scoring Summary

Task:
Requirement quality assessment using Deep NLP + Computer Vision + explainable neuro-symbolic scoring.

Inputs:
- C8 deep ambiguity predictions
- V7 page-level vision predictions
- V7 document-level vision summary

Deep model:
- RoBERTa-base quality classifier
- Labels: LOW_QUALITY, MEDIUM_QUALITY, HIGH_QUALITY
- Training strategy: silver-label learning from NLP and Computer Vision signals

Requirement-level output:
- Shape: {df_c9_final.shape}
- Average requirement quality score: {round(df_c9_final["requirement_quality_score"].mean(), 2)}
- Average deep quality confidence: {round(df_c9_final["deep_quality_confidence"].mean(), 4)}

Requirement quality level distribution:
{df_c9_final["requirement_quality_level"].value_counts().to_string()}

Deep quality label distribution:
{df_c9_final["deep_quality_label"].value_counts().to_string()}

Document quality level distribution:
{df_document_quality_summary["document_quality_level"].value_counts().to_string()}

Document review priority distribution:
{df_document_quality_summary["document_review_priority"].value_counts().to_string()}

Saved outputs:
- {c9_requirement_output_path}
- {c9_section_summary_path}
- {c9_document_summary_path}
- {c9_eval_results_path}
- {c9_manifest_path}
"""

with open(c9_summary_path, "w") as f:
    f.write(summary_text)

print(summary_text)
print("Saved:", c9_summary_path)

display(df_c9_manifest)

MODULE C9.18 — SAVE FINAL C9 OUTPUTS

C9 — Multimodal Requirement Quality Scoring Summary

Task:
Requirement quality assessment using Deep NLP + Computer Vision + explainable neuro-symbolic scoring.

Inputs:
- C8 deep ambiguity predictions
- V7 page-level vision predictions
- V7 document-level vision summary

Deep model:
- RoBERTa-base quality classifier
- Labels: LOW_QUALITY, MEDIUM_QUALITY, HIGH_QUALITY
- Training strategy: silver-label learning from NLP and Computer Vision signals

Requirement-level output:
- Shape: (3608, 91)
- Average requirement quality score: 86.28
- Average deep quality confidence: 0.8720999956130981

Requirement quality level distribution:
requirement_quality_level
EXCELLENT        2293
GOOD              913
REVIEW_NEEDED     344
WEAK               57
CRITICAL            1

Deep quality label distribution:
deep_quality_label
HIGH_QUALITY      2058
MEDIUM_QUALITY    1048
LOW_QUALITY        502

Document quality level distribution:
document_quality_level
EXCELLE

,output_name,path
0,requirement_level_quality_scores,/kaggle/working/c9_multimodal_requirement_qual...
1,section_quality_summary,/kaggle/working/c9_section_quality_summary.csv
2,document_quality_summary,/kaggle/working/c9_document_quality_summary.csv
3,roberta_quality_eval_results,/kaggle/working/c9_roberta_quality_eval_result...
4,quality_model_path,/kaggle/working/roberta_requirement_quality_model


In [21]:
# =========================================================
# MODULE C10.0 — LOAD C9 QUALITY OUTPUTS
# =========================================================

from pathlib import Path
import os
import re
import json
import pandas as pd
import numpy as np

print("=" * 80)
print("MODULE C10.0 — LOAD C9 QUALITY OUTPUTS")
print("=" * 80)

candidate_c9_paths = [
    Path("/kaggle/working/c9_multimodal_requirement_quality_scores.csv"),
    Path("/kaggle/input/datasets/cyrinemejriri/data/c9_multimodal_requirement_quality_scores.csv"),
    Path("/kaggle/input/datasets/cyrinemejriri/daataa/c9_multimodal_requirement_quality_scores.csv"),
]

# Search Kaggle input if needed
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "c9_multimodal_requirement_quality_scores.csv":
            candidate_c9_paths.append(Path(root) / f)

c9_path = None
for p in candidate_c9_paths:
    if p.exists():
        c9_path = p
        break

if c9_path is None:
    raise FileNotFoundError(
        "C9 output file not found. Please add Notebook 4 / C9 outputs as input."
    )

print("Selected C9 path:", c9_path)

df_c9 = pd.read_csv(c9_path)

print("C9 shape:", df_c9.shape)
print("Columns:")
print(df_c9.columns.tolist())

display(df_c9.head(10))

MODULE C10.0 — LOAD C9 QUALITY OUTPUTS
Selected C9 path: /kaggle/working/c9_multimodal_requirement_quality_scores.csv
C9 shape: (3608, 91)
Columns:
['doc_id', 'page_num', 'page_type', 'section_label', 'block_id', 'block_text', 'requirement_type_candidate', 'requirement_strength', 'requirement_confidence', 'deep_prediction', 'deep_confidence', 'deep_prediction_label', 'prediction_status', 'final_prediction', 'nfr_subtype_pred', 'nfr_subtype_confidence', 'nfr_subtype_model_name', 'ambiguity_pred_id', 'ambiguity_prediction', 'ambiguity_confidence', 'clear_probability', 'ambiguous_probability', 'ambiguity_model_name', 'ambiguity_status', 'final_ambiguity_label', 'true_page_type', 'vision_page_type', 'vision_confidence', 'vision_quality_flag', 'vision_prob_appendix_page', 'vision_prob_content_page', 'vision_prob_cover_page', 'vision_prob_low_text_page', 'vision_prob_toc_page', 'avg_vision_confidence', 'document_vision_quality_flag', 'low_confidence_pages', 'vision_accuracy_proxy', 'quality_

,doc_id,page_num,page_type,section_label,block_id,block_text,requirement_type_candidate,requirement_strength,requirement_confidence,deep_prediction,...,ambiguity_safety_score,visual_context_score,completeness_score,deep_quality_numeric_score,requirement_quality_score,requirement_quality_level,quality_issues,quality_recommendations,num_quality_issues,section_label_clean
0,0000 cctns,4,content_page,functional_requirements,9,"following investigation, police shall take the...",FR,strong,1.0,0,...,98.57,53.68,100.0,89.42,94.59,EXCELLENT,['vision_page_type_mismatch'],['Review page type consistency between NLP lab...,1,functional_requirements
1,0000 cctns,6,content_page,non_functional_requirements,9,The solution should provide detailed context-s...,NFR,medium,0.9,1,...,21.88,70.08,88.0,29.45,50.01,WEAK,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,7,non_functional_requirements
2,0000 cctns,6,content_page,non_functional_requirements,12,The help should be accessible to the users bot...,NFR,medium,0.9,1,...,10.81,70.08,88.0,27.90,54.69,WEAK,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,6,non_functional_requirements
3,0000 cctns,6,content_page,non_functional_requirements,15,The solution should provide an interface for t...,NFR,medium,0.9,1,...,97.00,70.08,100.0,85.81,85.67,EXCELLENT,"['nfr_missing_numeric_target', 'nfr_missing_th...",['Add a measurable numeric target such as resp...,2,non_functional_requirements
4,0000 cctns,6,content_page,non_functional_requirements,18,"The solution should send alerts (e.g., email, ...",NFR,medium,0.9,0,...,96.42,70.08,100.0,88.79,95.68,EXCELLENT,[],['No major quality issue detected.'],0,non_functional_requirements
5,0000 cctns,6,content_page,non_functional_requirements,21,The solution should enable the user to track t...,NFR,medium,0.9,0,...,70.35,70.08,100.0,86.60,89.12,EXCELLENT,['ambiguity_uncertain_review'],['No major quality issue detected.'],1,non_functional_requirements
6,0000 cctns,6,content_page,non_functional_requirements,24,The solution should enable the help-desk user ...,NFR,medium,0.9,0,...,84.78,70.08,100.0,59.13,92.29,EXCELLENT,[],['No major quality issue detected.'],0,non_functional_requirements
7,0000 cctns,6,content_page,non_functional_requirements,28,The support solution should be accessible to t...,NFR,medium,0.9,1,...,10.70,70.08,88.0,27.95,58.79,REVIEW_NEEDED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,5,non_functional_requirements
8,0000 cctns,6,content_page,non_functional_requirements,35,System must keep an unalterable audit trail ca...,NFR,strong,1.0,1,...,98.66,70.08,100.0,59.51,84.88,GOOD,"['nfr_missing_numeric_target', 'nfr_missing_th...",['Add a measurable numeric target such as resp...,2,non_functional_requirements
9,0000 cctns,8,content_page,non_functional_requirements,5,The System must allow a user to be a member of...,NFR,strong,1.0,0,...,99.06,85.79,100.0,88.98,88.97,EXCELLENT,"['fr_nfr_classification_uncertain', 'low_fr_nf...",['Review whether this requirement is functiona...,2,non_functional_requirements


In [22]:
# =========================================================
# MODULE C10.1 — SELECT REQUIREMENTS FOR LLM REWRITE
# =========================================================

print("=" * 80)
print("MODULE C10.1 — SELECT REQUIREMENTS FOR LLM REWRITE")
print("=" * 80)

required_cols = [
    "doc_id",
    "page_num",
    "block_text",
    "requirement_quality_score",
    "requirement_quality_level",
    "quality_issues",
    "quality_recommendations",
    "final_prediction",
    "final_ambiguity_label",
    "vision_page_type",
    "vision_confidence"
]

for col in required_cols:
    if col not in df_c9.columns:
        raise ValueError(f"Missing required C9 column: {col}")

rewrite_levels = ["CRITICAL", "WEAK", "REVIEW_NEEDED"]

df_c10_candidates = df_c9[
    df_c9["requirement_quality_level"].isin(rewrite_levels)
].copy()

df_c10_candidates = df_c10_candidates.sort_values(
    by=["requirement_quality_score", "deep_quality_confidence"],
    ascending=[True, True]
).reset_index(drop=True)

print("Total requirements:", len(df_c9))
print("Candidates for rewrite:", len(df_c10_candidates))

print("\nRewrite candidate level distribution:")
print(df_c10_candidates["requirement_quality_level"].value_counts())

display(df_c10_candidates[
    [
        "doc_id",
        "page_num",
        "block_text",
        "requirement_quality_score",
        "requirement_quality_level",
        "final_prediction",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence",
        "quality_issues",
        "quality_recommendations"
    ]
].head(20))

MODULE C10.1 — SELECT REQUIREMENTS FOR LLM REWRITE
Total requirements: 3608
Candidates for rewrite: 402

Rewrite candidate level distribution:
requirement_quality_level
REVIEW_NEEDED    344
WEAK              57
CRITICAL           1
Name: count, dtype: int64


,doc_id,page_num,block_text,requirement_quality_score,requirement_quality_level,final_prediction,final_ambiguity_label,vision_page_type,vision_confidence,quality_issues,quality_recommendations
0,0000 cctns,11,should be sufficient so as not to impede reada...,39.44,CRITICAL,UNCERTAIN,AMBIGUOUS,content_page,0.374997,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
1,2002 evla corr,14,messages should be able to easily filter the e...,42.36,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.973620,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
2,2001 beyond,19,• The authoring tool should support easy devel...,43.93,WEAK,UNCERTAIN,AMBIGUOUS,appendix_page,0.585477,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
3,2001 beyond,25,• The UI Editor itself should be a powerful an...,44.24,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.620956,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
4,0000 cctns scanned,12,"Use of ""white space ""White space"" On page Le. ...",44.92,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.699740,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
5,1995 gemini,40,"Normally, raw data will be acquired and stored...",45.14,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.990939,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
6,2003 pnnl,34,tower fan should be off,45.47,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.998112,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
7,1999 tcs,99,The TCS reliability will be considered in ever...,46.54,WEAK,UNCERTAIN,AMBIGUOUS,appendix_page,0.432609,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
8,0000 cctns,11,recover from errors should be minimized.,46.54,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.374997,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...
9,2003 agentmom,4,This project will be a framework that provides...,46.56,WEAK,UNCERTAIN,AMBIGUOUS,appendix_page,0.474599,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...


In [23]:
# =========================================================
# MODULE C10.2 — CONTROLLED REWRITE SAMPLE
# =========================================================

MAX_REWRITE_SAMPLES = 100

df_c10_sample = df_c10_candidates.head(MAX_REWRITE_SAMPLES).copy()
df_c10_sample = df_c10_sample.reset_index(drop=True)

print("Rewrite sample shape:", df_c10_sample.shape)

display(df_c10_sample[
    [
        "doc_id",
        "page_num",
        "block_text",
        "requirement_quality_score",
        "requirement_quality_level",
        "quality_issues"
    ]
].head(20))

Rewrite sample shape: (100, 91)


,doc_id,page_num,block_text,requirement_quality_score,requirement_quality_level,quality_issues
0,0000 cctns,11,should be sufficient so as not to impede reada...,39.44,CRITICAL,"['ambiguous_requirement', 'deep_model_low_qual..."
1,2002 evla corr,14,messages should be able to easily filter the e...,42.36,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."
2,2001 beyond,19,• The authoring tool should support easy devel...,43.93,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."
3,2001 beyond,25,• The UI Editor itself should be a powerful an...,44.24,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."
4,0000 cctns scanned,12,"Use of ""white space ""White space"" On page Le. ...",44.92,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."
5,1995 gemini,40,"Normally, raw data will be acquired and stored...",45.14,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."
6,2003 pnnl,34,tower fan should be off,45.47,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."
7,1999 tcs,99,The TCS reliability will be considered in ever...,46.54,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."
8,0000 cctns,11,recover from errors should be minimized.,46.54,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."
9,2003 agentmom,4,This project will be a framework that provides...,46.56,WEAK,"['ambiguous_requirement', 'deep_model_low_qual..."


In [24]:
# =========================================================
# MODULE C10.3 — LOAD MODERN INSTRUCTION-TUNED LLM
# =========================================================

print("=" * 80)
print("MODULE C10.3 — LOAD MODERN INSTRUCTION-TUNED LLM")
print("=" * 80)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Recommended safe model for Kaggle CPU/GPU
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Selected LLM:", LLM_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True
)

if device == "cpu":
    llm_model = llm_model.to(device)

llm_model.eval()

print("LLM loaded successfully.")

MODULE C10.3 — LOAD MODERN INSTRUCTION-TUNED LLM
Device: cpu
Selected LLM: Qwen/Qwen2.5-1.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded successfully.


In [25]:
# =========================================================
# MODULE C10.4 — BUILD MULTIMODAL REWRITE PROMPT
# =========================================================

print("=" * 80)
print("MODULE C10.4 — BUILD MULTIMODAL REWRITE PROMPT")
print("=" * 80)

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x)

def build_requirement_rewrite_prompt(row):
    original_req = safe_str(row["block_text"])
    quality_level = safe_str(row["requirement_quality_level"])
    quality_score = safe_str(row["requirement_quality_score"])
    issues = safe_str(row["quality_issues"])
    recommendations = safe_str(row["quality_recommendations"])
    fr_nfr = safe_str(row["final_prediction"])
    ambiguity = safe_str(row["final_ambiguity_label"])
    vision_page_type = safe_str(row["vision_page_type"])
    vision_confidence = safe_str(row["vision_confidence"])
    section = safe_str(row.get("section_label", "unknown_section"))

    prompt = f"""
You are an expert Software Requirements Engineer.

Your task is to rewrite a weak or ambiguous software requirement into a clear, testable, measurable, and unambiguous requirement.

Original requirement:
{original_req}

Context:
- Requirement type prediction: {fr_nfr}
- Section label: {section}
- Ambiguity label: {ambiguity}
- Quality level: {quality_level}
- Quality score: {quality_score}
- Detected quality issues: {issues}
- Suggested recommendations: {recommendations}
- Visual page type from computer vision: {vision_page_type}
- Vision confidence: {vision_confidence}

Rewrite rules:
1. Preserve the original meaning.
2. Do not invent new features.
3. Use clear requirement language.
4. Prefer the word "shall".
5. Make the requirement testable and measurable.
6. Remove vague words such as appropriate, sufficient, user-friendly, fast, easy, etc.
7. If a numeric threshold is missing but necessary, use a placeholder such as [SPECIFY THRESHOLD].
8. Output only valid JSON.

Return this JSON format:
{{
  "improved_requirement": "...",
  "rewrite_strategy": "...",
  "what_was_fixed": ["...", "..."],
  "remaining_assumptions": ["...", "..."]
}}
"""
    return prompt.strip()

example_prompt = build_requirement_rewrite_prompt(df_c10_sample.iloc[0])
print(example_prompt[:3000])

MODULE C10.4 — BUILD MULTIMODAL REWRITE PROMPT
You are an expert Software Requirements Engineer.

Your task is to rewrite a weak or ambiguous software requirement into a clear, testable, measurable, and unambiguous requirement.

Original requirement:
should be sufficient so as not to impede readability.

Context:
- Requirement type prediction: UNCERTAIN
- Section label: non_functional_requirements
- Ambiguity label: AMBIGUOUS
- Quality level: CRITICAL
- Quality score: 39.44
- Detected quality issues: ['ambiguous_requirement', 'deep_model_low_quality_risk', 'fr_nfr_classification_uncertain', 'low_fr_nfr_confidence', 'low_nfr_subtype_confidence', 'low_visual_confidence', 'missing_actor_or_system_context', 'nfr_missing_numeric_target', 'nfr_missing_threshold', 'vague_language']
- Suggested recommendations: ['Rewrite the requirement to remove ambiguity and make the expected behavior explicit.', 'Replace vague terms with precise measurable wording.', 'Add a measurable numeric target such as

In [26]:
# =========================================================
# MODULE C10.5 — LLM GENERATION FUNCTION
# =========================================================

print("=" * 80)
print("MODULE C10.5 — LLM GENERATION FUNCTION")
print("=" * 80)

def generate_llm_response(prompt, max_new_tokens=350):
    messages = [
        {
            "role": "system",
            "content": "You are a senior software requirements engineer specialized in SRS quality improvement."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return response.strip()


def extract_json_from_response(response):
    try:
        start = response.find("{")
        end = response.rfind("}") + 1

        if start == -1 or end == 0:
            return {
                "improved_requirement": response.strip(),
                "rewrite_strategy": "free_text_generation",
                "what_was_fixed": [],
                "remaining_assumptions": []
            }

        json_text = response[start:end]
        return json.loads(json_text)

    except Exception:
        return {
            "improved_requirement": response.strip(),
            "rewrite_strategy": "json_parse_failed",
            "what_was_fixed": [],
            "remaining_assumptions": []
        }

print("Generation utilities ready.")

MODULE C10.5 — LLM GENERATION FUNCTION
Generation utilities ready.


In [27]:
# =========================================================
# MODULE C10.6 — GENERATE IMPROVED REQUIREMENTS
# =========================================================

print("=" * 80)
print("MODULE C10.6 — GENERATE IMPROVED REQUIREMENTS")
print("=" * 80)

from tqdm.auto import tqdm

rewrite_rows = []

for idx, row in tqdm(df_c10_sample.iterrows(), total=len(df_c10_sample), desc="Generating rewrites"):
    prompt = build_requirement_rewrite_prompt(row)

    try:
        raw_response = generate_llm_response(prompt)
        parsed = extract_json_from_response(raw_response)

        improved_requirement = parsed.get("improved_requirement", "")
        rewrite_strategy = parsed.get("rewrite_strategy", "")
        what_was_fixed = parsed.get("what_was_fixed", [])
        remaining_assumptions = parsed.get("remaining_assumptions", [])

    except Exception as e:
        raw_response = str(e)
        improved_requirement = ""
        rewrite_strategy = "generation_failed"
        what_was_fixed = []
        remaining_assumptions = []

    rewrite_rows.append({
        "doc_id": row["doc_id"],
        "page_num": row["page_num"],
        "original_requirement": row["block_text"],
        "original_quality_score": row["requirement_quality_score"],
        "original_quality_level": row["requirement_quality_level"],
        "final_prediction": row["final_prediction"],
        "final_ambiguity_label": row["final_ambiguity_label"],
        "vision_page_type": row["vision_page_type"],
        "vision_confidence": row["vision_confidence"],
        "quality_issues": row["quality_issues"],
        "quality_recommendations": row["quality_recommendations"],
        "improved_requirement": improved_requirement,
        "rewrite_strategy": rewrite_strategy,
        "what_was_fixed": what_was_fixed,
        "remaining_assumptions": remaining_assumptions,
        "raw_llm_response": raw_response
    })

df_c10_rewrites = pd.DataFrame(rewrite_rows)

print("C10 rewrite output shape:", df_c10_rewrites.shape)

display(df_c10_rewrites[
    [
        "doc_id",
        "page_num",
        "original_quality_level",
        "original_quality_score",
        "original_requirement",
        "improved_requirement",
        "rewrite_strategy"
    ]
].head(20))

MODULE C10.6 — GENERATE IMPROVED REQUIREMENTS


Generating rewrites:   0%|          | 0/100 [00:00<?, ?it/s]

C10 rewrite output shape: (100, 16)


,doc_id,page_num,original_quality_level,original_quality_score,original_requirement,improved_requirement,rewrite_strategy
0,0000 cctns,11,CRITICAL,39.44,should be sufficient so as not to impede reada...,The system shall ensure that it does not imped...,[Remove vague terms like 'appropriate' and 'su...
1,2002 evla corr,14,WEAK,42.36,messages should be able to easily filter the e...,The system shall allow for filtering of error ...,Removed ambiguity by specifying that the syste...
2,2001 beyond,19,WEAK,43.93,• The authoring tool should support easy devel...,The authoring tool shall provide tools for cre...,[Removed vague terms like 'appropriate' and 's...
3,2001 beyond,25,WEAK,44.24,• The UI Editor itself should be a powerful an...,The UI Editor shall be designed to provide a r...,Removed vague terms like 'appropriate' and 'su...
4,0000 cctns scanned,12,WEAK,44.92,"Use of ""white space ""White space"" On page Le. ...",The distance between blocks of information dis...,Removed vague terms like 'appropriate' and 'su...
5,1995 gemini,40,WEAK,45.14,"Normally, raw data will be acquired and stored...",The system shall acquire and store raw data in...,[Remove vague terms like 'quick look' and repl...
6,2003 pnnl,34,WEAK,45.47,tower fan should be off,The tower fan shall be turned off.,[Removed vague words like 'appropriate' and 's...
7,1999 tcs,99,WEAK,46.54,The TCS reliability will be considered in ever...,All phases of the design and development proce...,Removed ambiguity by specifying that all phase...
8,0000 cctns,11,WEAK,46.54,recover from errors should be minimized.,The recovery process for errors shall minimize...,Removed ambiguity by specifying that the recov...
9,2003 agentmom,4,WEAK,46.56,This project will be a framework that provides...,The framework shall provide reusable agents wi...,Removed ambiguity by specifying the nature of ...


In [28]:
# =========================================================
# MODULE C10.7 — VALIDATE GENERATED REQUIREMENTS
# =========================================================

print("=" * 80)
print("MODULE C10.7 — VALIDATE GENERATED REQUIREMENTS")
print("=" * 80)

vague_terms = [
    "fast", "quick", "easy", "simple", "user-friendly", "appropriate",
    "sufficient", "adequate", "as needed", "as required", "etc",
    "efficient", "robust", "seamless", "good", "bad", "normal"
]

requirement_keywords = [
    "shall", "must", "should", "will"
]

numeric_pattern = r"\d+|%|seconds?|minutes?|hours?|days?|ms|milliseconds?|kb|mb|gb|users?|requests?|transactions?"

def validate_requirement_text(text):
    text = safe_str(text)
    low = text.lower()

    word_count = len(low.split())

    has_requirement_keyword = any(k in low for k in requirement_keywords)
    has_vague_term = any(term in low for term in vague_terms)
    has_numeric_constraint = bool(re.search(numeric_pattern, low))
    has_placeholder = "[specify" in low or "specify threshold" in low

    clarity_score = 100
    if word_count < 6:
        clarity_score -= 35
    if has_vague_term:
        clarity_score -= 25
    if not has_requirement_keyword:
        clarity_score -= 20

    testability_score = 100
    if not has_requirement_keyword:
        testability_score -= 20
    if has_vague_term:
        testability_score -= 25
    if word_count < 8:
        testability_score -= 25

    measurability_score = 100
    if not has_numeric_constraint and not has_placeholder:
        measurability_score -= 35
    if has_vague_term:
        measurability_score -= 25

    completeness_score = 100
    if word_count < 8:
        completeness_score -= 35
    if not has_requirement_keyword:
        completeness_score -= 20

    final_validation_score = np.mean([
        clarity_score,
        testability_score,
        measurability_score,
        completeness_score
    ])

    final_validation_score = max(0, min(100, final_validation_score))

    if final_validation_score >= 85:
        validation_label = "STRONG_REWRITE"
    elif final_validation_score >= 70:
        validation_label = "ACCEPTABLE_REWRITE"
    elif final_validation_score >= 50:
        validation_label = "NEEDS_REVIEW"
    else:
        validation_label = "FAILED_REWRITE"

    return {
        "rewrite_word_count": word_count,
        "rewrite_has_requirement_keyword": has_requirement_keyword,
        "rewrite_has_vague_term": has_vague_term,
        "rewrite_has_numeric_constraint": has_numeric_constraint,
        "rewrite_has_placeholder": has_placeholder,
        "rewrite_clarity_score": round(clarity_score, 2),
        "rewrite_testability_score": round(testability_score, 2),
        "rewrite_measurability_score": round(measurability_score, 2),
        "rewrite_completeness_score": round(completeness_score, 2),
        "rewrite_validation_score": round(final_validation_score, 2),
        "rewrite_validation_label": validation_label
    }

validation_rows = df_c10_rewrites["improved_requirement"].apply(validate_requirement_text)
df_validation = pd.DataFrame(validation_rows.tolist())

df_c10_validated = pd.concat(
    [df_c10_rewrites.reset_index(drop=True), df_validation.reset_index(drop=True)],
    axis=1
)

print("Validated rewrite shape:", df_c10_validated.shape)

print("\nRewrite validation distribution:")
print(df_c10_validated["rewrite_validation_label"].value_counts())

display(df_c10_validated[
    [
        "original_quality_level",
        "original_quality_score",
        "original_requirement",
        "improved_requirement",
        "rewrite_validation_score",
        "rewrite_validation_label",
        "rewrite_has_vague_term",
        "rewrite_has_numeric_constraint",
        "rewrite_has_placeholder"
    ]
].head(30))

MODULE C10.7 — VALIDATE GENERATED REQUIREMENTS
Validated rewrite shape: (100, 27)

Rewrite validation distribution:
rewrite_validation_label
STRONG_REWRITE        76
ACCEPTABLE_REWRITE    24
Name: count, dtype: int64


,original_quality_level,original_quality_score,original_requirement,improved_requirement,rewrite_validation_score,rewrite_validation_label,rewrite_has_vague_term,rewrite_has_numeric_constraint,rewrite_has_placeholder
0,CRITICAL,39.44,should be sufficient so as not to impede reada...,The system shall ensure that it does not imped...,91.25,STRONG_REWRITE,False,False,False
1,WEAK,42.36,messages should be able to easily filter the e...,The system shall allow for filtering of error ...,91.25,STRONG_REWRITE,False,False,False
2,WEAK,43.93,• The authoring tool should support easy devel...,The authoring tool shall provide tools for cre...,100.00,STRONG_REWRITE,False,True,False
3,WEAK,44.24,• The UI Editor itself should be a powerful an...,The UI Editor shall be designed to provide a r...,72.50,ACCEPTABLE_REWRITE,True,False,False
4,WEAK,44.92,"Use of ""white space ""White space"" On page Le. ...",The distance between blocks of information dis...,72.50,ACCEPTABLE_REWRITE,True,False,False
5,WEAK,45.14,"Normally, raw data will be acquired and stored...",The system shall acquire and store raw data in...,72.50,ACCEPTABLE_REWRITE,True,False,False
6,WEAK,45.47,tower fan should be off,The tower fan shall be turned off.,76.25,ACCEPTABLE_REWRITE,False,False,False
7,WEAK,46.54,The TCS reliability will be considered in ever...,All phases of the design and development proce...,91.25,STRONG_REWRITE,False,False,False
8,WEAK,46.54,recover from errors should be minimized.,The recovery process for errors shall minimize...,100.00,STRONG_REWRITE,False,True,False
9,WEAK,46.56,This project will be a framework that provides...,The framework shall provide reusable agents wi...,91.25,STRONG_REWRITE,False,False,False


In [29]:
# =========================================================
# MODULE C10.8 — BEFORE / AFTER IMPROVEMENT ANALYSIS
# =========================================================

print("=" * 80)
print("MODULE C10.8 — BEFORE / AFTER IMPROVEMENT ANALYSIS")
print("=" * 80)

df_c10_validated["quality_score_improvement_proxy"] = (
    df_c10_validated["rewrite_validation_score"] 
    - df_c10_validated["original_quality_score"].astype(float)
)

def improvement_flag(delta):
    if delta >= 20:
        return "MAJOR_IMPROVEMENT"
    elif delta >= 5:
        return "MODERATE_IMPROVEMENT"
    elif delta >= -5:
        return "STABLE_OR_MINOR_CHANGE"
    else:
        return "POSSIBLE_REGRESSION"

df_c10_validated["rewrite_improvement_flag"] = (
    df_c10_validated["quality_score_improvement_proxy"].apply(improvement_flag)
)

print("Improvement distribution:")
print(df_c10_validated["rewrite_improvement_flag"].value_counts())

print("\nAverage original quality score:", round(df_c10_validated["original_quality_score"].mean(), 2))
print("Average rewrite validation score:", round(df_c10_validated["rewrite_validation_score"].mean(), 2))
print("Average improvement:", round(df_c10_validated["quality_score_improvement_proxy"].mean(), 2))

display(df_c10_validated[
    [
        "original_quality_level",
        "original_quality_score",
        "rewrite_validation_score",
        "quality_score_improvement_proxy",
        "rewrite_improvement_flag",
        "original_requirement",
        "improved_requirement"
    ]
].head(30))

MODULE C10.8 — BEFORE / AFTER IMPROVEMENT ANALYSIS
Improvement distribution:
rewrite_improvement_flag
MAJOR_IMPROVEMENT       95
MODERATE_IMPROVEMENT     5
Name: count, dtype: int64

Average original quality score: 53.28
Average rewrite validation score: 90.32
Average improvement: 37.04


,original_quality_level,original_quality_score,rewrite_validation_score,quality_score_improvement_proxy,rewrite_improvement_flag,original_requirement,improved_requirement
0,CRITICAL,39.44,91.25,51.81,MAJOR_IMPROVEMENT,should be sufficient so as not to impede reada...,The system shall ensure that it does not imped...
1,WEAK,42.36,91.25,48.89,MAJOR_IMPROVEMENT,messages should be able to easily filter the e...,The system shall allow for filtering of error ...
2,WEAK,43.93,100.00,56.07,MAJOR_IMPROVEMENT,• The authoring tool should support easy devel...,The authoring tool shall provide tools for cre...
3,WEAK,44.24,72.50,28.26,MAJOR_IMPROVEMENT,• The UI Editor itself should be a powerful an...,The UI Editor shall be designed to provide a r...
4,WEAK,44.92,72.50,27.58,MAJOR_IMPROVEMENT,"Use of ""white space ""White space"" On page Le. ...",The distance between blocks of information dis...
5,WEAK,45.14,72.50,27.36,MAJOR_IMPROVEMENT,"Normally, raw data will be acquired and stored...",The system shall acquire and store raw data in...
6,WEAK,45.47,76.25,30.78,MAJOR_IMPROVEMENT,tower fan should be off,The tower fan shall be turned off.
7,WEAK,46.54,91.25,44.71,MAJOR_IMPROVEMENT,The TCS reliability will be considered in ever...,All phases of the design and development proce...
8,WEAK,46.54,100.00,53.46,MAJOR_IMPROVEMENT,recover from errors should be minimized.,The recovery process for errors shall minimize...
9,WEAK,46.56,91.25,44.69,MAJOR_IMPROVEMENT,This project will be a framework that provides...,The framework shall provide reusable agents wi...


In [30]:
# =========================================================
# MODULE C10.9 — SAVE C10 OUTPUTS
# =========================================================

print("=" * 80)
print("MODULE C10.9 — SAVE C10 OUTPUTS")
print("=" * 80)

c10_rewrite_path = "/kaggle/working/c10_llm_requirement_rewrites.csv"
c10_summary_path = "/kaggle/working/c10_llm_rewrite_summary.txt"

df_c10_validated.to_csv(c10_rewrite_path, index=False)

c10_summary = f"""
C10 — LLM-Based Requirement Improvement Summary

Task:
Rewrite weak, ambiguous, or low-quality SRS requirements using a modern instruction-tuned LLM.

Input:
- C9 multimodal requirement quality scores
- NLP signals: FR/NFR, ambiguity, quality issues
- Computer Vision signals: page type, vision confidence, visual quality flag

Model:
- {LLM_MODEL_NAME}

Generation strategy:
- Instruction-tuned LLM generation
- JSON structured output
- Requirement engineering constraints
- Preserve original meaning
- Improve clarity, testability, measurability, and completeness

Validation:
- Rule-based post-generation validation
- Checks requirement keyword, vagueness, measurability, completeness
- Computes rewrite validation score

Total candidates available:
{len(df_c10_candidates)}

Generated rewrites:
{len(df_c10_validated)}

Rewrite validation distribution:
{df_c10_validated["rewrite_validation_label"].value_counts().to_string()}

Rewrite improvement distribution:
{df_c10_validated["rewrite_improvement_flag"].value_counts().to_string()}

Average original quality score:
{round(df_c10_validated["original_quality_score"].mean(), 2)}

Average rewrite validation score:
{round(df_c10_validated["rewrite_validation_score"].mean(), 2)}

Average improvement proxy:
{round(df_c10_validated["quality_score_improvement_proxy"].mean(), 2)}

Saved output:
- {c10_rewrite_path}
"""

print(c10_summary)

with open(c10_summary_path, "w") as f:
    f.write(c10_summary)

print("Saved:", c10_rewrite_path)
print("Saved:", c10_summary_path)

MODULE C10.9 — SAVE C10 OUTPUTS

C10 — LLM-Based Requirement Improvement Summary

Task:
Rewrite weak, ambiguous, or low-quality SRS requirements using a modern instruction-tuned LLM.

Input:
- C9 multimodal requirement quality scores
- NLP signals: FR/NFR, ambiguity, quality issues
- Computer Vision signals: page type, vision confidence, visual quality flag

Model:
- Qwen/Qwen2.5-1.5B-Instruct

Generation strategy:
- Instruction-tuned LLM generation
- JSON structured output
- Requirement engineering constraints
- Preserve original meaning
- Improve clarity, testability, measurability, and completeness

Validation:
- Rule-based post-generation validation
- Checks requirement keyword, vagueness, measurability, completeness
- Computes rewrite validation score

Total candidates available:
402

Generated rewrites:
100

Rewrite validation distribution:
rewrite_validation_label
STRONG_REWRITE        76
ACCEPTABLE_REWRITE    24

Rewrite improvement distribution:
rewrite_improvement_flag
MAJOR_

# =========================================================
# MODULE C11 — FINAL INTEGRATED MULTIMODAL SRS REPORT
# =========================================================

In [31]:
# =========================================================
# MODULE C11.0 — LOAD FINAL PIPELINE OUTPUTS
# =========================================================

from pathlib import Path
import os
import re
import ast
import json
import pandas as pd
import numpy as np

print("=" * 80)
print("MODULE C11.0 — LOAD FINAL PIPELINE OUTPUTS")
print("=" * 80)

def find_file(filename, search_roots=None):
    if search_roots is None:
        search_roots = [
            Path("/kaggle/working"),
            Path("/kaggle/input")
        ]
    
    # Direct check
    for root in search_roots:
        direct = root / filename
        if direct.exists():
            return direct
    
    # Recursive search
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if len(matches) > 0:
                return matches[0]
    
    return None


required_files = {
    "c9_quality": "c9_multimodal_requirement_quality_scores.csv",
    "c9_document_summary": "c9_document_quality_summary.csv",
    "c9_section_summary": "c9_section_quality_summary.csv",
    "c10_rewrites": "c10_llm_requirement_rewrites.csv",
}

found_paths = {}

for key, filename in required_files.items():
    path = find_file(filename)
    found_paths[key] = path
    print(f"{key}: {filename}")
    print(" ->", path)
    print()

critical_keys = ["c9_quality", "c9_document_summary", "c10_rewrites"]

for key in critical_keys:
    if found_paths[key] is None:
        raise FileNotFoundError(
            f"Missing required file for C11: {required_files[key]}. "
            "Please make sure C9 and C10 outputs are available."
        )

print("All critical C11 files found.")

MODULE C11.0 — LOAD FINAL PIPELINE OUTPUTS
c9_quality: c9_multimodal_requirement_quality_scores.csv
 -> /kaggle/working/c9_multimodal_requirement_quality_scores.csv

c9_document_summary: c9_document_quality_summary.csv
 -> /kaggle/working/c9_document_quality_summary.csv

c9_section_summary: c9_section_quality_summary.csv
 -> /kaggle/working/c9_section_quality_summary.csv

c10_rewrites: c10_llm_requirement_rewrites.csv
 -> /kaggle/working/c10_llm_requirement_rewrites.csv

All critical C11 files found.


In [32]:
# =========================================================
# MODULE C11.1 — READ FINAL DATASETS
# =========================================================

df_c9_quality = pd.read_csv(found_paths["c9_quality"])
df_c9_doc_summary = pd.read_csv(found_paths["c9_document_summary"])
df_c10_rewrites = pd.read_csv(found_paths["c10_rewrites"])

if found_paths["c9_section_summary"] is not None:
    df_c9_section_summary = pd.read_csv(found_paths["c9_section_summary"])
else:
    df_c9_section_summary = pd.DataFrame()

print("C9 requirement quality shape:", df_c9_quality.shape)
print("C9 document summary shape:", df_c9_doc_summary.shape)
print("C9 section summary shape:", df_c9_section_summary.shape)
print("C10 rewrites shape:", df_c10_rewrites.shape)

print("\nC9 quality columns:")
print(df_c9_quality.columns.tolist())

print("\nC10 rewrite columns:")
print(df_c10_rewrites.columns.tolist())

display(df_c9_quality.head(5))
display(df_c9_doc_summary.head(5))
display(df_c10_rewrites.head(5))

C9 requirement quality shape: (3608, 91)
C9 document summary shape: (24, 29)
C9 section summary shape: (58, 23)
C10 rewrites shape: (100, 29)

C9 quality columns:
['doc_id', 'page_num', 'page_type', 'section_label', 'block_id', 'block_text', 'requirement_type_candidate', 'requirement_strength', 'requirement_confidence', 'deep_prediction', 'deep_confidence', 'deep_prediction_label', 'prediction_status', 'final_prediction', 'nfr_subtype_pred', 'nfr_subtype_confidence', 'nfr_subtype_model_name', 'ambiguity_pred_id', 'ambiguity_prediction', 'ambiguity_confidence', 'clear_probability', 'ambiguous_probability', 'ambiguity_model_name', 'ambiguity_status', 'final_ambiguity_label', 'true_page_type', 'vision_page_type', 'vision_confidence', 'vision_quality_flag', 'vision_prob_appendix_page', 'vision_prob_content_page', 'vision_prob_cover_page', 'vision_prob_low_text_page', 'vision_prob_toc_page', 'avg_vision_confidence', 'document_vision_quality_flag', 'low_confidence_pages', 'vision_accuracy_pr

,doc_id,page_num,page_type,section_label,block_id,block_text,requirement_type_candidate,requirement_strength,requirement_confidence,deep_prediction,...,ambiguity_safety_score,visual_context_score,completeness_score,deep_quality_numeric_score,requirement_quality_score,requirement_quality_level,quality_issues,quality_recommendations,num_quality_issues,section_label_clean
0,0000 cctns,4,content_page,functional_requirements,9,"following investigation, police shall take the...",FR,strong,1.0,0,...,98.57,53.68,100.0,89.42,94.59,EXCELLENT,['vision_page_type_mismatch'],['Review page type consistency between NLP lab...,1,functional_requirements
1,0000 cctns,6,content_page,non_functional_requirements,9,The solution should provide detailed context-s...,NFR,medium,0.9,1,...,21.88,70.08,88.0,29.45,50.01,WEAK,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,7,non_functional_requirements
2,0000 cctns,6,content_page,non_functional_requirements,12,The help should be accessible to the users bot...,NFR,medium,0.9,1,...,10.81,70.08,88.0,27.90,54.69,WEAK,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,6,non_functional_requirements
3,0000 cctns,6,content_page,non_functional_requirements,15,The solution should provide an interface for t...,NFR,medium,0.9,1,...,97.00,70.08,100.0,85.81,85.67,EXCELLENT,"['nfr_missing_numeric_target', 'nfr_missing_th...",['Add a measurable numeric target such as resp...,2,non_functional_requirements
4,0000 cctns,6,content_page,non_functional_requirements,18,"The solution should send alerts (e.g., email, ...",NFR,medium,0.9,0,...,96.42,70.08,100.0,88.79,95.68,EXCELLENT,[],['No major quality issue detected.'],0,non_functional_requirements


,doc_id,total_requirements,document_quality_score,avg_clarity_score,avg_testability_score,avg_measurability_score,avg_classification_reliability_score,avg_ambiguity_safety_score,avg_visual_context_score,avg_completeness_score,...,uncertain_fr_nfr_requirements,low_vision_confidence_requirements,vision_page_mismatch_requirements,document_vision_quality_flag,document_quality_level,critical_ratio,weak_or_critical_ratio,ambiguous_ratio,vision_mismatch_ratio,document_review_priority
0,0000 cctns,109,71.42,77.52,77.25,72.71,78.55,48.14,67.43,94.39,...,29,48,1,moderate_visual_structure,GOOD,0.009,0.138,0.541,0.009,HIGH_PRIORITY_REVIEW
1,0000 cctns scanned,33,74.82,74.24,80.00,76.82,78.59,55.60,79.52,98.18,...,9,0,0,moderate_visual_structure,GOOD,0.000,0.121,0.455,0.000,HIGH_PRIORITY_REVIEW
2,0000 gamma j,123,88.58,95.49,93.33,83.82,85.44,90.43,78.96,96.29,...,21,3,18,moderate_visual_structure,EXCELLENT,0.000,0.000,0.081,0.146,LOW_PRIORITY_REVIEW
3,0000 gamma j scanned,56,89.71,92.50,93.21,87.95,93.46,89.39,75.75,97.43,...,3,6,12,moderate_visual_structure,EXCELLENT,0.000,0.018,0.071,0.214,LOW_PRIORITY_REVIEW
4,0000 inventory,10,75.39,86.00,84.00,64.00,93.02,60.41,56.72,95.20,...,0,1,5,moderate_visual_structure,GOOD,0.000,0.100,0.400,0.500,HIGH_PRIORITY_REVIEW


,doc_id,page_num,original_requirement,original_quality_score,original_quality_level,final_prediction,final_ambiguity_label,vision_page_type,vision_confidence,quality_issues,...,rewrite_has_numeric_constraint,rewrite_has_placeholder,rewrite_clarity_score,rewrite_testability_score,rewrite_measurability_score,rewrite_completeness_score,rewrite_validation_score,rewrite_validation_label,quality_score_improvement_proxy,rewrite_improvement_flag
0,0000 cctns,11,should be sufficient so as not to impede reada...,39.44,CRITICAL,UNCERTAIN,AMBIGUOUS,content_page,0.374997,"['ambiguous_requirement', 'deep_model_low_qual...",...,False,False,100,100,65,100,91.25,STRONG_REWRITE,51.81,MAJOR_IMPROVEMENT
1,2002 evla corr,14,messages should be able to easily filter the e...,42.36,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.973620,"['ambiguous_requirement', 'deep_model_low_qual...",...,False,False,100,100,65,100,91.25,STRONG_REWRITE,48.89,MAJOR_IMPROVEMENT
2,2001 beyond,19,• The authoring tool should support easy devel...,43.93,WEAK,UNCERTAIN,AMBIGUOUS,appendix_page,0.585477,"['ambiguous_requirement', 'deep_model_low_qual...",...,True,False,100,100,100,100,100.00,STRONG_REWRITE,56.07,MAJOR_IMPROVEMENT
3,2001 beyond,25,• The UI Editor itself should be a powerful an...,44.24,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.620956,"['ambiguous_requirement', 'deep_model_low_qual...",...,False,False,75,75,40,100,72.50,ACCEPTABLE_REWRITE,28.26,MAJOR_IMPROVEMENT
4,0000 cctns scanned,12,"Use of ""white space ""White space"" On page Le. ...",44.92,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.699740,"['ambiguous_requirement', 'deep_model_low_qual...",...,False,False,75,75,40,100,72.50,ACCEPTABLE_REWRITE,27.58,MAJOR_IMPROVEMENT


In [33]:
# =========================================================
# MODULE C11.2 — NORMALIZE FINAL MERGE KEYS
# =========================================================

print("=" * 80)
print("MODULE C11.2 — NORMALIZE FINAL MERGE KEYS")
print("=" * 80)

def normalize_doc_id(x):
    return (
        str(x)
        .lower()
        .replace("-", " ")
        .replace("_", " ")
        .strip()
    )

def normalize_text(x):
    x = str(x)
    x = re.sub(r"\s+", " ", x)
    return x.strip().lower()

for df in [df_c9_quality, df_c9_doc_summary, df_c10_rewrites]:
    if "doc_id" in df.columns:
        df["doc_id"] = df["doc_id"].apply(normalize_doc_id)

if "page_num" in df_c9_quality.columns:
    df_c9_quality["page_num"] = df_c9_quality["page_num"].astype(int)

if "page_num" in df_c10_rewrites.columns:
    df_c10_rewrites["page_num"] = df_c10_rewrites["page_num"].astype(int)

# Make sure C10 has a normalized original requirement column
if "original_requirement" not in df_c10_rewrites.columns:
    if "block_text" in df_c10_rewrites.columns:
        df_c10_rewrites["original_requirement"] = df_c10_rewrites["block_text"]
    else:
        raise ValueError("C10 rewrites must contain original_requirement or block_text.")

df_c9_quality["requirement_text_norm"] = df_c9_quality["block_text"].apply(normalize_text)
df_c10_rewrites["requirement_text_norm"] = df_c10_rewrites["original_requirement"].apply(normalize_text)

print("C9 unique docs:", df_c9_quality["doc_id"].nunique())
print("C10 unique docs:", df_c10_rewrites["doc_id"].nunique())

print("\nC9 page range:", df_c9_quality["page_num"].min(), "->", df_c9_quality["page_num"].max())
print("C10 page range:", df_c10_rewrites["page_num"].min(), "->", df_c10_rewrites["page_num"].max())

MODULE C11.2 — NORMALIZE FINAL MERGE KEYS
C9 unique docs: 24
C10 unique docs: 19

C9 page range: 2 -> 176
C10 page range: 4 -> 172


In [34]:
# =========================================================
# MODULE C11.3 — MERGE LLM REWRITES INTO FINAL REQUIREMENT TABLE
# =========================================================

print("=" * 80)
print("MODULE C11.3 — MERGE LLM REWRITES")
print("=" * 80)

rewrite_cols = [
    "doc_id",
    "page_num",
    "requirement_text_norm",
    "original_requirement",
    "improved_requirement",
    "rewrite_strategy",
    "what_was_fixed",
    "remaining_assumptions",
    "rewrite_validation_score",
    "rewrite_validation_label",
    "rewrite_improvement_flag",
    "quality_score_improvement_proxy"
]

rewrite_cols = [c for c in rewrite_cols if c in df_c10_rewrites.columns]

df_c11_requirements = pd.merge(
    df_c9_quality,
    df_c10_rewrites[rewrite_cols],
    on=["doc_id", "page_num", "requirement_text_norm"],
    how="left"
)

df_c11_requirements["has_llm_rewrite"] = df_c11_requirements["improved_requirement"].notna()

print("C11 final requirement table shape:", df_c11_requirements.shape)
print("\nLLM rewrite coverage:")
print(df_c11_requirements["has_llm_rewrite"].value_counts())

print("\nRequirement quality distribution:")
print(df_c11_requirements["requirement_quality_level"].value_counts(dropna=False))

display(df_c11_requirements[
    [
        "doc_id",
        "page_num",
        "block_text",
        "requirement_quality_score",
        "requirement_quality_level",
        "final_prediction",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence",
        "has_llm_rewrite",
        "improved_requirement"
    ]
].head(20))

MODULE C11.3 — MERGE LLM REWRITES
C11 final requirement table shape: (3608, 102)

LLM rewrite coverage:
has_llm_rewrite
False    3508
True      100
Name: count, dtype: int64

Requirement quality distribution:
requirement_quality_level
EXCELLENT        2293
GOOD              913
REVIEW_NEEDED     344
WEAK               57
CRITICAL            1
Name: count, dtype: int64


,doc_id,page_num,block_text,requirement_quality_score,requirement_quality_level,final_prediction,final_ambiguity_label,vision_page_type,vision_confidence,has_llm_rewrite,improved_requirement
0,0000 cctns,4,"following investigation, police shall take the...",94.59,EXCELLENT,FR,CLEAR,appendix_page,0.617763,False,NaN
1,0000 cctns,6,The solution should provide detailed context-s...,50.01,WEAK,UNCERTAIN,AMBIGUOUS,content_page,0.664405,True,"```json\n{\n ""improved_requirement"": ""The sol..."
2,0000 cctns,6,The help should be accessible to the users bot...,54.69,WEAK,NFR,AMBIGUOUS,content_page,0.664405,True,The help feature shall be available through bo...
3,0000 cctns,6,The solution should provide an interface for t...,85.67,EXCELLENT,NFR,CLEAR,content_page,0.664405,False,NaN
4,0000 cctns,6,"The solution should send alerts (e.g., email, ...",95.68,EXCELLENT,FR,CLEAR,content_page,0.664405,False,NaN
5,0000 cctns,6,The solution should enable the user to track t...,89.12,EXCELLENT,FR,UNCERTAIN,content_page,0.664405,False,NaN
6,0000 cctns,6,The solution should enable the help-desk user ...,92.29,EXCELLENT,FR,CLEAR,content_page,0.664405,False,NaN
7,0000 cctns,6,The support solution should be accessible to t...,58.79,REVIEW_NEEDED,NFR,AMBIGUOUS,content_page,0.664405,True,The support solution shall be accessible to th...
8,0000 cctns,6,System must keep an unalterable audit trail ca...,84.88,GOOD,NFR,CLEAR,content_page,0.664405,False,NaN
9,0000 cctns,8,The System must allow a user to be a member of...,88.97,EXCELLENT,UNCERTAIN,CLEAR,content_page,0.921359,False,NaN


In [35]:
# =========================================================
# MODULE C11.4 — FINAL REQUIREMENT ACTION PRIORITY
# =========================================================

print("=" * 80)
print("MODULE C11.4 — FINAL REQUIREMENT ACTION PRIORITY")
print("=" * 80)

def assign_requirement_action(row):
    level = str(row.get("requirement_quality_level", "")).upper()
    ambiguity = str(row.get("final_ambiguity_label", "")).upper()
    confidence = float(row.get("deep_quality_confidence", 0))
    vision_conf = float(row.get("vision_confidence", 1))
    has_rewrite = bool(row.get("has_llm_rewrite", False))
    
    if level == "CRITICAL":
        return "URGENT_REWRITE"
    
    if level == "WEAK" and ambiguity == "AMBIGUOUS":
        return "REWRITE_REQUIRED"
    
    if level == "WEAK":
        return "REVIEW_AND_REWRITE"
    
    if level == "REVIEW_NEEDED":
        return "MANUAL_REVIEW"
    
    if confidence < 0.65:
        return "MODEL_UNCERTAINTY_REVIEW"
    
    if vision_conf < 0.60:
        return "SOURCE_PAGE_VISUAL_REVIEW"
    
    if has_rewrite:
        return "REWRITE_AVAILABLE"
    
    return "ACCEPTABLE"


df_c11_requirements["final_action_priority"] = df_c11_requirements.apply(
    assign_requirement_action,
    axis=1
)

print("Final action priority distribution:")
print(df_c11_requirements["final_action_priority"].value_counts())

display(df_c11_requirements[
    [
        "doc_id",
        "page_num",
        "block_text",
        "requirement_quality_score",
        "requirement_quality_level",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence",
        "final_action_priority",
        "has_llm_rewrite",
        "improved_requirement"
    ]
].head(30))

MODULE C11.4 — FINAL REQUIREMENT ACTION PRIORITY
Final action priority distribution:
final_action_priority
ACCEPTABLE                   2493
SOURCE_PAGE_VISUAL_REVIEW     394
MANUAL_REVIEW                 344
MODEL_UNCERTAINTY_REVIEW      319
REWRITE_REQUIRED               57
URGENT_REWRITE                  1
Name: count, dtype: int64


,doc_id,page_num,block_text,requirement_quality_score,requirement_quality_level,final_ambiguity_label,vision_page_type,vision_confidence,final_action_priority,has_llm_rewrite,improved_requirement
0,0000 cctns,4,"following investigation, police shall take the...",94.59,EXCELLENT,CLEAR,appendix_page,0.617763,ACCEPTABLE,False,NaN
1,0000 cctns,6,The solution should provide detailed context-s...,50.01,WEAK,AMBIGUOUS,content_page,0.664405,REWRITE_REQUIRED,True,"```json\n{\n ""improved_requirement"": ""The sol..."
2,0000 cctns,6,The help should be accessible to the users bot...,54.69,WEAK,AMBIGUOUS,content_page,0.664405,REWRITE_REQUIRED,True,The help feature shall be available through bo...
3,0000 cctns,6,The solution should provide an interface for t...,85.67,EXCELLENT,CLEAR,content_page,0.664405,ACCEPTABLE,False,NaN
4,0000 cctns,6,"The solution should send alerts (e.g., email, ...",95.68,EXCELLENT,CLEAR,content_page,0.664405,ACCEPTABLE,False,NaN
5,0000 cctns,6,The solution should enable the user to track t...,89.12,EXCELLENT,UNCERTAIN,content_page,0.664405,ACCEPTABLE,False,NaN
6,0000 cctns,6,The solution should enable the help-desk user ...,92.29,EXCELLENT,CLEAR,content_page,0.664405,ACCEPTABLE,False,NaN
7,0000 cctns,6,The support solution should be accessible to t...,58.79,REVIEW_NEEDED,AMBIGUOUS,content_page,0.664405,MANUAL_REVIEW,True,The support solution shall be accessible to th...
8,0000 cctns,6,System must keep an unalterable audit trail ca...,84.88,GOOD,CLEAR,content_page,0.664405,ACCEPTABLE,False,NaN
9,0000 cctns,8,The System must allow a user to be a member of...,88.97,EXCELLENT,CLEAR,content_page,0.921359,ACCEPTABLE,False,NaN


In [36]:
# =========================================================
# MODULE C11.5 — BUILD DOCUMENT-LEVEL FINAL REPORT TABLE
# =========================================================

print("=" * 80)
print("MODULE C11.5 — DOCUMENT-LEVEL FINAL REPORT TABLE")
print("=" * 80)

doc_req_summary = (
    df_c11_requirements
    .groupby("doc_id")
    .agg(
        total_requirements=("block_text", "count"),
        avg_requirement_quality_score=("requirement_quality_score", "mean"),
        excellent_requirements=("requirement_quality_level", lambda x: (x == "EXCELLENT").sum()),
        good_requirements=("requirement_quality_level", lambda x: (x == "GOOD").sum()),
        review_needed_requirements=("requirement_quality_level", lambda x: (x == "REVIEW_NEEDED").sum()),
        weak_requirements=("requirement_quality_level", lambda x: (x == "WEAK").sum()),
        critical_requirements=("requirement_quality_level", lambda x: (x == "CRITICAL").sum()),
        ambiguous_requirements=("final_ambiguity_label", lambda x: (x == "AMBIGUOUS").sum()),
        uncertain_ambiguity_requirements=("final_ambiguity_label", lambda x: (x == "UNCERTAIN").sum()),
        low_vision_confidence_requirements=("vision_quality_flag", lambda x: (x == "low_confidence_visual_prediction").sum()),
        vision_page_mismatch_requirements=("vision_page_mismatch", lambda x: x.sum() if x.dtype != "object" else (x == True).sum()),
        llm_rewrites_available=("has_llm_rewrite", "sum")
    )
    .reset_index()
)

doc_req_summary["avg_requirement_quality_score"] = doc_req_summary["avg_requirement_quality_score"].round(2)

doc_req_summary["weak_or_critical_requirements"] = (
    doc_req_summary["weak_requirements"] + doc_req_summary["critical_requirements"]
)

doc_req_summary["weak_or_critical_ratio"] = (
    doc_req_summary["weak_or_critical_requirements"] /
    doc_req_summary["total_requirements"].replace(0, np.nan)
).round(3)

doc_req_summary["ambiguity_ratio"] = (
    doc_req_summary["ambiguous_requirements"] /
    doc_req_summary["total_requirements"].replace(0, np.nan)
).round(3)

doc_req_summary["rewrite_coverage_ratio"] = (
    doc_req_summary["llm_rewrites_available"] /
    doc_req_summary["weak_or_critical_requirements"].replace(0, np.nan)
).fillna(0).round(3)


def assign_final_document_status(row):
    if row["critical_requirements"] > 0:
        return "CRITICAL_DOCUMENT_REVIEW"
    if row["weak_or_critical_ratio"] >= 0.20:
        return "HIGH_PRIORITY_REVIEW"
    if row["ambiguity_ratio"] >= 0.30:
        return "AMBIGUITY_REVIEW"
    if row["avg_requirement_quality_score"] >= 85:
        return "HIGH_QUALITY_DOCUMENT"
    if row["avg_requirement_quality_score"] >= 75:
        return "GOOD_DOCUMENT"
    return "MODERATE_REVIEW"


doc_req_summary["final_document_status"] = doc_req_summary.apply(
    assign_final_document_status,
    axis=1
)

# Merge with existing C9 document summary if available
df_c11_document_report = pd.merge(
    df_c9_doc_summary,
    doc_req_summary,
    on="doc_id",
    how="right",
    suffixes=("_c9", "")
)

print("Final document report shape:", df_c11_document_report.shape)

print("\nFinal document status distribution:")
print(df_c11_document_report["final_document_status"].value_counts())

display(df_c11_document_report.head(30))

MODULE C11.5 — DOCUMENT-LEVEL FINAL REPORT TABLE
Final document report shape: (24, 46)

Final document status distribution:
final_document_status
HIGH_QUALITY_DOCUMENT       14
GOOD_DOCUMENT                5
AMBIGUITY_REVIEW             4
CRITICAL_DOCUMENT_REVIEW     1
Name: count, dtype: int64


,doc_id,total_requirements_c9,document_quality_score,avg_clarity_score,avg_testability_score,avg_measurability_score,avg_classification_reliability_score,avg_ambiguity_safety_score,avg_visual_context_score,avg_completeness_score,...,ambiguous_requirements,uncertain_ambiguity_requirements,low_vision_confidence_requirements,vision_page_mismatch_requirements,llm_rewrites_available,weak_or_critical_requirements,weak_or_critical_ratio,ambiguity_ratio,rewrite_coverage_ratio,final_document_status
0,0000 cctns,109,71.42,77.52,77.25,72.71,78.55,48.14,67.43,94.39,...,59,9,48,1,31,15,0.138,0.541,2.067,CRITICAL_DOCUMENT_REVIEW
1,0000 cctns scanned,33,74.82,74.24,80.00,76.82,78.59,55.60,79.52,98.18,...,15,3,0,0,4,4,0.121,0.455,1.000,AMBIGUITY_REVIEW
2,0000 gamma j,123,88.58,95.49,93.33,83.82,85.44,90.43,78.96,96.29,...,10,2,3,18,3,0,0.000,0.081,0.000,HIGH_QUALITY_DOCUMENT
3,0000 gamma j scanned,56,89.71,92.50,93.21,87.95,93.46,89.39,75.75,97.43,...,4,3,6,12,1,1,0.018,0.071,1.000,HIGH_QUALITY_DOCUMENT
4,0000 inventory,10,75.39,86.00,84.00,64.00,93.02,60.41,56.72,95.20,...,4,0,1,5,3,1,0.100,0.400,3.000,AMBIGUITY_REVIEW
5,0000 inventory scanned,7,74.34,79.29,82.86,82.86,80.99,52.02,69.13,78.29,...,3,1,0,3,2,0,0.000,0.429,0.000,AMBIGUITY_REVIEW
6,1995 gemini,424,82.51,88.24,83.86,83.87,78.75,72.23,91.23,90.44,...,102,46,2,0,20,13,0.031,0.241,1.538,GOOD_DOCUMENT
7,1998 themas,117,89.49,91.79,88.12,98.72,94.53,79.44,85.63,91.78,...,19,12,7,0,0,0,0.000,0.162,0.000,HIGH_QUALITY_DOCUMENT
8,1999 tcs,1107,88.86,97.84,91.85,88.50,83.46,95.47,71.41,91.78,...,22,25,230,202,7,5,0.005,0.020,1.400,HIGH_QUALITY_DOCUMENT
9,2001 beyond,100,78.71,83.20,81.70,92.40,85.62,61.18,60.66,91.55,...,34,18,38,29,8,4,0.040,0.340,2.000,AMBIGUITY_REVIEW


In [37]:
# =========================================================
# MODULE C11.6 — TOP PROBLEMATIC REQUIREMENTS
# =========================================================

print("=" * 80)
print("MODULE C11.6 — TOP PROBLEMATIC REQUIREMENTS")
print("=" * 80)

priority_order = {
    "URGENT_REWRITE": 0,
    "REWRITE_REQUIRED": 1,
    "REVIEW_AND_REWRITE": 2,
    "MANUAL_REVIEW": 3,
    "MODEL_UNCERTAINTY_REVIEW": 4,
    "SOURCE_PAGE_VISUAL_REVIEW": 5,
    "REWRITE_AVAILABLE": 6,
    "ACCEPTABLE": 7
}

df_c11_requirements["priority_rank"] = df_c11_requirements["final_action_priority"].map(priority_order).fillna(99)

df_top_problematic_requirements = (
    df_c11_requirements
    .sort_values(
        by=["priority_rank", "requirement_quality_score", "deep_quality_confidence"],
        ascending=[True, True, True]
    )
    .copy()
)

print("Top problematic requirements shape:", df_top_problematic_requirements.shape)

display(df_top_problematic_requirements[
    [
        "doc_id",
        "page_num",
        "block_text",
        "requirement_quality_score",
        "requirement_quality_level",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence",
        "final_action_priority",
        "quality_issues",
        "quality_recommendations",
        "improved_requirement"
    ]
].head(50))

MODULE C11.6 — TOP PROBLEMATIC REQUIREMENTS
Top problematic requirements shape: (3608, 104)


,doc_id,page_num,block_text,requirement_quality_score,requirement_quality_level,final_ambiguity_label,vision_page_type,vision_confidence,final_action_priority,quality_issues,quality_recommendations,improved_requirement
37,0000 cctns,11,should be sufficient so as not to impede reada...,39.44,CRITICAL,AMBIGUOUS,content_page,0.374997,URGENT_REWRITE,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The system shall ensure that it does not imped...
2965,2002 evla corr,14,messages should be able to easily filter the e...,42.36,WEAK,AMBIGUOUS,content_page,0.973620,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The system shall allow for filtering of error ...
2010,2001 beyond,19,• The authoring tool should support easy devel...,43.93,WEAK,AMBIGUOUS,appendix_page,0.585477,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The authoring tool shall provide tools for cre...
2032,2001 beyond,25,• The UI Editor itself should be a powerful an...,44.24,WEAK,AMBIGUOUS,content_page,0.620956,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The UI Editor shall be designed to provide a r...
126,0000 cctns scanned,12,"Use of ""white space ""White space"" On page Le. ...",44.92,WEAK,AMBIGUOUS,content_page,0.699740,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The distance between blocks of information dis...
520,1995 gemini,40,"Normally, raw data will be acquired and stored...",45.14,WEAK,AMBIGUOUS,content_page,0.990939,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The system shall acquire and store raw data in...
3231,2003 pnnl,34,tower fan should be off,45.47,WEAK,AMBIGUOUS,content_page,0.998112,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The tower fan shall be turned off.
1739,1999 tcs,99,The TCS reliability will be considered in ever...,46.54,WEAK,AMBIGUOUS,appendix_page,0.432609,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,All phases of the design and development proce...
33,0000 cctns,11,recover from errors should be minimized.,46.54,WEAK,AMBIGUOUS,content_page,0.374997,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The recovery process for errors shall minimize...
3100,2003 agentmom,4,This project will be a framework that provides...,46.56,WEAK,AMBIGUOUS,appendix_page,0.474599,REWRITE_REQUIRED,"['ambiguous_requirement', 'deep_model_low_qual...",['Rewrite the requirement to remove ambiguity ...,The framework shall provide reusable agents wi...


In [38]:
# =========================================================
# MODULE C11.7 — GENERATE MARKDOWN REPORT PER DOCUMENT
# =========================================================

print("=" * 80)
print("MODULE C11.7 — GENERATE MARKDOWN REPORT PER DOCUMENT")
print("=" * 80)

reports_dir = Path("/kaggle/working/final_srs_reports")
reports_dir.mkdir(parents=True, exist_ok=True)

def safe_filename(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


def short_text(x, max_len=500):
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    if len(x) > max_len:
        return x[:max_len] + "..."
    return x


def format_list_cell(x):
    if pd.isna(x):
        return "None"
    
    if isinstance(x, list):
        items = x
    else:
        text = str(x)
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                items = parsed
            else:
                items = [text]
        except Exception:
            items = [text]
    
    if len(items) == 0:
        return "None"
    
    return "\n".join([f"- {str(item)}" for item in items[:8]])


generated_reports = []

for doc_id in sorted(df_c11_document_report["doc_id"].unique()):
    doc_row = df_c11_document_report[df_c11_document_report["doc_id"] == doc_id].iloc[0]
    doc_reqs = df_c11_requirements[df_c11_requirements["doc_id"] == doc_id].copy()
    
    doc_reqs_sorted = doc_reqs.sort_values(
        by=["priority_rank", "requirement_quality_score"],
        ascending=[True, True]
    )
    
    top_issues = doc_reqs_sorted.head(10)
    rewrites = doc_reqs_sorted[doc_reqs_sorted["has_llm_rewrite"] == True].head(10)
    
    report_md = f"""# Final Multimodal SRS Intelligence Report

## Document

**Document ID:** `{doc_id}`

## Executive Summary

| Metric | Value |
|---|---:|
| Total requirements | {int(doc_row.get("total_requirements", 0))} |
| Average requirement quality score | {round(float(doc_row.get("avg_requirement_quality_score", 0)), 2)} |
| Excellent requirements | {int(doc_row.get("excellent_requirements", 0))} |
| Good requirements | {int(doc_row.get("good_requirements", 0))} |
| Review-needed requirements | {int(doc_row.get("review_needed_requirements", 0))} |
| Weak requirements | {int(doc_row.get("weak_requirements", 0))} |
| Critical requirements | {int(doc_row.get("critical_requirements", 0))} |
| Ambiguous requirements | {int(doc_row.get("ambiguous_requirements", 0))} |
| LLM rewrites available | {int(doc_row.get("llm_rewrites_available", 0))} |

**Final document status:** `{doc_row.get("final_document_status", "UNKNOWN")}`

## Multimodal Intelligence Used

This report combines:

- NLP requirement extraction and FR/NFR classification.
- NFR subtype classification.
- Ambiguity detection.
- Computer Vision page classification.
- ResNet18 visual page understanding.
- Grad-CAM explainability.
- Deep requirement quality scoring.
- LLM-based requirement rewriting.

## Top Problematic Requirements

"""

    for idx, row in top_issues.iterrows():
        report_md += f"""
### Requirement on page {row.get("page_num", "N/A")}

**Original requirement**

> {short_text(row.get("block_text", ""), 700)}

**Quality**

- Quality score: `{round(float(row.get("requirement_quality_score", 0)), 2)}`
- Quality level: `{row.get("requirement_quality_level", "UNKNOWN")}`
- Final action: `{row.get("final_action_priority", "UNKNOWN")}`
- Ambiguity label: `{row.get("final_ambiguity_label", "UNKNOWN")}`
- Vision page type: `{row.get("vision_page_type", "UNKNOWN")}`
- Vision confidence: `{round(float(row.get("vision_confidence", 0)), 3)}`

**Detected issues**

{format_list_cell(row.get("quality_issues", ""))}

**Recommendations**

{format_list_cell(row.get("quality_recommendations", ""))}
"""

        if row.get("has_llm_rewrite", False):
            report_md += f"""
**LLM-improved requirement**

> {short_text(row.get("improved_requirement", ""), 700)}

**Rewrite validation**

- Validation score: `{round(float(row.get("rewrite_validation_score", 0)), 2) if not pd.isna(row.get("rewrite_validation_score", np.nan)) else "N/A"}`
- Validation label: `{row.get("rewrite_validation_label", "N/A")}`
- Improvement flag: `{row.get("rewrite_improvement_flag", "N/A")}`
"""

    report_md += """

## Conclusion

This document has been analyzed using a multimodal AI pipeline combining document image understanding, deep NLP, explainability, quality scoring, and LLM-based improvement.

The generated outputs can support requirement review, quality assurance, ambiguity reduction, and SRS modernization.
"""

    report_path = reports_dir / f"{safe_filename(doc_id)}_final_srs_report.md"
    
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_md)
    
    generated_reports.append({
        "doc_id": doc_id,
        "report_path": str(report_path)
    })

df_generated_reports = pd.DataFrame(generated_reports)

print("Generated reports:", len(df_generated_reports))
display(df_generated_reports.head(30))

MODULE C11.7 — GENERATE MARKDOWN REPORT PER DOCUMENT
Generated reports: 24


,doc_id,report_path
0,0000 cctns,/kaggle/working/final_srs_reports/0000_cctns_f...
1,0000 cctns scanned,/kaggle/working/final_srs_reports/0000_cctns_s...
2,0000 gamma j,/kaggle/working/final_srs_reports/0000_gamma_j...
3,0000 gamma j scanned,/kaggle/working/final_srs_reports/0000_gamma_j...
4,0000 inventory,/kaggle/working/final_srs_reports/0000_invento...
5,0000 inventory scanned,/kaggle/working/final_srs_reports/0000_invento...
6,1995 gemini,/kaggle/working/final_srs_reports/1995_gemini_...
7,1998 themas,/kaggle/working/final_srs_reports/1998_themas_...
8,1999 tcs,/kaggle/working/final_srs_reports/1999_tcs_fin...
9,2001 beyond,/kaggle/working/final_srs_reports/2001_beyond_...


In [39]:
# =========================================================
# MODULE C11.8 — SAVE FINAL INTEGRATED OUTPUTS
# =========================================================

print("=" * 80)
print("MODULE C11.8 — SAVE FINAL INTEGRATED OUTPUTS")
print("=" * 80)

c11_requirement_report_path = "/kaggle/working/c11_final_requirement_report.csv"
c11_document_report_path = "/kaggle/working/c11_final_document_report.csv"
c11_top_problematic_path = "/kaggle/working/c11_top_problematic_requirements.csv"
c11_generated_reports_path = "/kaggle/working/c11_generated_report_paths.csv"

df_c11_requirements.to_csv(c11_requirement_report_path, index=False)
df_c11_document_report.to_csv(c11_document_report_path, index=False)
df_top_problematic_requirements.to_csv(c11_top_problematic_path, index=False)
df_generated_reports.to_csv(c11_generated_reports_path, index=False)

print("Saved:", c11_requirement_report_path)
print("Saved:", c11_document_report_path)
print("Saved:", c11_top_problematic_path)
print("Saved:", c11_generated_reports_path)

c11_manifest = pd.DataFrame([
    {"output_name": "final_requirement_report", "path": c11_requirement_report_path},
    {"output_name": "final_document_report", "path": c11_document_report_path},
    {"output_name": "top_problematic_requirements", "path": c11_top_problematic_path},
    {"output_name": "generated_markdown_reports", "path": str(reports_dir)},
    {"output_name": "generated_report_paths", "path": c11_generated_reports_path},
])

c11_manifest_path = "/kaggle/working/c11_outputs_manifest.csv"
c11_manifest.to_csv(c11_manifest_path, index=False)

display(c11_manifest)

print("Saved:", c11_manifest_path)

MODULE C11.8 — SAVE FINAL INTEGRATED OUTPUTS
Saved: /kaggle/working/c11_final_requirement_report.csv
Saved: /kaggle/working/c11_final_document_report.csv
Saved: /kaggle/working/c11_top_problematic_requirements.csv
Saved: /kaggle/working/c11_generated_report_paths.csv


,output_name,path
0,final_requirement_report,/kaggle/working/c11_final_requirement_report.csv
1,final_document_report,/kaggle/working/c11_final_document_report.csv
2,top_problematic_requirements,/kaggle/working/c11_top_problematic_requiremen...
3,generated_markdown_reports,/kaggle/working/final_srs_reports
4,generated_report_paths,/kaggle/working/c11_generated_report_paths.csv


Saved: /kaggle/working/c11_outputs_manifest.csv


In [40]:
# =========================================================
# MODULE C11.9 — FINAL PROJECT SUMMARY
# =========================================================

print("=" * 80)
print("MODULE C11.9 — FINAL PROJECT SUMMARY")
print("=" * 80)

total_requirements = len(df_c11_requirements)
total_documents = df_c11_requirements["doc_id"].nunique()
avg_quality = df_c11_requirements["requirement_quality_score"].mean()

quality_dist = df_c11_requirements["requirement_quality_level"].value_counts()
action_dist = df_c11_requirements["final_action_priority"].value_counts()
doc_status_dist = df_c11_document_report["final_document_status"].value_counts()

llm_rewrites = int(df_c11_requirements["has_llm_rewrite"].sum())

final_summary = f"""
Final Multimodal SRS Intelligence System Summary

Project:
A multimodal AI system for Software Requirements Specification analysis.

Input:
- Raw SRS PDF documents
- Page images extracted from PDF documents
- OCR / extracted text
- Requirement blocks

Main AI Components:
1. Document structure intelligence
2. Requirement extraction
3. DistilBERT FR/NFR classification
4. RoBERTa NFR subtype classification
5. RoBERTa ambiguity detection with LIME XAI
6. Custom CNN from scratch for page image classification
7. Pretrained ResNet18 for page image classification
8. Grad-CAM visual explainability
9. Multimodal requirement quality scoring
10. Qwen2.5 instruction-tuned LLM requirement rewriting
11. Final integrated multimodal reporting

Final Dataset:
- Documents analyzed: {total_documents}
- Requirements analyzed: {total_requirements}
- Average requirement quality score: {avg_quality:.2f}
- LLM rewrites integrated: {llm_rewrites}

Requirement Quality Distribution:
{quality_dist.to_string()}

Final Action Priority Distribution:
{action_dist.to_string()}

Document Status Distribution:
{doc_status_dist.to_string()}

Final Outputs:
- {c11_requirement_report_path}
- {c11_document_report_path}
- {c11_top_problematic_path}
- {c11_generated_reports_path}
- {reports_dir}

Conclusion:
The final system combines Computer Vision, NLP, Deep Learning, XAI, multimodal scoring, and LLM generation to support intelligent SRS review and improvement.
"""

print(final_summary)

final_summary_path = "/kaggle/working/c11_final_project_summary.txt"

with open(final_summary_path, "w", encoding="utf-8") as f:
    f.write(final_summary)

print("Saved:", final_summary_path)

MODULE C11.9 — FINAL PROJECT SUMMARY

Final Multimodal SRS Intelligence System Summary

Project:
A multimodal AI system for Software Requirements Specification analysis.

Input:
- Raw SRS PDF documents
- Page images extracted from PDF documents
- OCR / extracted text
- Requirement blocks

Main AI Components:
1. Document structure intelligence
2. Requirement extraction
3. DistilBERT FR/NFR classification
4. RoBERTa NFR subtype classification
5. RoBERTa ambiguity detection with LIME XAI
6. Custom CNN from scratch for page image classification
7. Pretrained ResNet18 for page image classification
8. Grad-CAM visual explainability
9. Multimodal requirement quality scoring
10. Qwen2.5 instruction-tuned LLM requirement rewriting
11. Final integrated multimodal reporting

Final Dataset:
- Documents analyzed: 24
- Requirements analyzed: 3608
- Average requirement quality score: 86.28
- LLM rewrites integrated: 100

Requirement Quality Distribution:
requirement_quality_level
EXCELLENT        229

In [41]:
# ============================================================
# MODULE C12.0 — LOAD FINAL INTEGRATED OUTPUTS
# ============================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("MODULE C12.0 — LOAD FINAL INTEGRATED OUTPUTS")
print("=" * 80)

C11_CANDIDATES = {
    "final_requirement_report": [
        "/kaggle/working/c11_final_requirement_report.csv",
        "/kaggle/input/c11-final-integrated-outputs/c11_final_requirement_report.csv",
    ],
    "final_document_report": [
        "/kaggle/working/c11_final_document_report.csv",
        "/kaggle/input/c11-final-integrated-outputs/c11_final_document_report.csv",
    ],
    "top_problematic_requirements": [
        "/kaggle/working/c11_top_problematic_requirements.csv",
        "/kaggle/input/c11-final-integrated-outputs/c11_top_problematic_requirements.csv",
    ],
    "generated_report_paths": [
        "/kaggle/working/c11_generated_report_paths.csv",
        "/kaggle/input/c11-final-integrated-outputs/c11_generated_report_paths.csv",
    ],
}

def first_existing_path(paths):
    for p in paths:
        if Path(p).exists():
            return p
    return None

selected_c12_paths = {}
for key, paths in C11_CANDIDATES.items():
    selected_c12_paths[key] = first_existing_path(paths)
    print(f"{key}: {selected_c12_paths[key]}")

for key, path in selected_c12_paths.items():
    if path is None:
        raise FileNotFoundError(
            f"Missing required C11 output: {key}. "
            "Please run Module C11 first or add C11 outputs as Kaggle input datasets."
        )

df_req = pd.read_csv(selected_c12_paths["final_requirement_report"])
df_doc = pd.read_csv(selected_c12_paths["final_document_report"])
df_top = pd.read_csv(selected_c12_paths["top_problematic_requirements"])
df_paths = pd.read_csv(selected_c12_paths["generated_report_paths"])

print("\nLoaded shapes:")
print("Requirement report:", df_req.shape)
print("Document report:", df_doc.shape)
print("Top problematic:", df_top.shape)
print("Generated reports:", df_paths.shape)

display(df_req.head(5))
display(df_doc.head(5))

MODULE C12.0 — LOAD FINAL INTEGRATED OUTPUTS
final_requirement_report: /kaggle/working/c11_final_requirement_report.csv
final_document_report: /kaggle/working/c11_final_document_report.csv
top_problematic_requirements: /kaggle/working/c11_top_problematic_requirements.csv
generated_report_paths: /kaggle/working/c11_generated_report_paths.csv

Loaded shapes:
Requirement report: (3608, 104)
Document report: (24, 46)
Top problematic: (3608, 104)
Generated reports: (24, 2)


,doc_id,page_num,page_type,section_label,block_id,block_text,requirement_type_candidate,requirement_strength,requirement_confidence,deep_prediction,...,rewrite_strategy,what_was_fixed,remaining_assumptions,rewrite_validation_score,rewrite_validation_label,rewrite_improvement_flag,quality_score_improvement_proxy,has_llm_rewrite,final_action_priority,priority_rank
0,0000 cctns,4,content_page,functional_requirements,9,"following investigation, police shall take the...",FR,strong,1.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,ACCEPTABLE,7
1,0000 cctns,6,content_page,non_functional_requirements,9,The solution should provide detailed context-s...,NFR,medium,0.9,1,...,json_parse_failed,[],[],81.25,ACCEPTABLE_REWRITE,MAJOR_IMPROVEMENT,31.24,True,REWRITE_REQUIRED,1
2,0000 cctns,6,content_page,non_functional_requirements,12,The help should be accessible to the users bot...,NFR,medium,0.9,1,...,Removed vague terms like 'appropriate' and 'su...,"['ambiguous_requirement', 'deep_model_low_qual...",['Ensure that the help feature is designed wit...,91.25,STRONG_REWRITE,MAJOR_IMPROVEMENT,36.56,True,REWRITE_REQUIRED,1
3,0000 cctns,6,content_page,non_functional_requirements,15,The solution should provide an interface for t...,NFR,medium,0.9,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,ACCEPTABLE,7
4,0000 cctns,6,content_page,non_functional_requirements,18,"The solution should send alerts (e.g., email, ...",NFR,medium,0.9,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,ACCEPTABLE,7


,doc_id,total_requirements_c9,document_quality_score,avg_clarity_score,avg_testability_score,avg_measurability_score,avg_classification_reliability_score,avg_ambiguity_safety_score,avg_visual_context_score,avg_completeness_score,...,ambiguous_requirements,uncertain_ambiguity_requirements,low_vision_confidence_requirements,vision_page_mismatch_requirements,llm_rewrites_available,weak_or_critical_requirements,weak_or_critical_ratio,ambiguity_ratio,rewrite_coverage_ratio,final_document_status
0,0000 cctns,109,71.42,77.52,77.25,72.71,78.55,48.14,67.43,94.39,...,59,9,48,1,31,15,0.138,0.541,2.067,CRITICAL_DOCUMENT_REVIEW
1,0000 cctns scanned,33,74.82,74.24,80.00,76.82,78.59,55.60,79.52,98.18,...,15,3,0,0,4,4,0.121,0.455,1.000,AMBIGUITY_REVIEW
2,0000 gamma j,123,88.58,95.49,93.33,83.82,85.44,90.43,78.96,96.29,...,10,2,3,18,3,0,0.000,0.081,0.000,HIGH_QUALITY_DOCUMENT
3,0000 gamma j scanned,56,89.71,92.50,93.21,87.95,93.46,89.39,75.75,97.43,...,4,3,6,12,1,1,0.018,0.071,1.000,HIGH_QUALITY_DOCUMENT
4,0000 inventory,10,75.39,86.00,84.00,64.00,93.02,60.41,56.72,95.20,...,4,0,1,5,3,1,0.100,0.400,3.000,AMBIGUITY_REVIEW


In [42]:
# ============================================================
# MODULE C12.1 — BUILD DASHBOARD ANALYTICS SUMMARY
# ============================================================

print("=" * 80)
print("MODULE C12.1 — BUILD DASHBOARD ANALYTICS SUMMARY")
print("=" * 80)

def safe_col(df, col, default=np.nan):
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df))

summary_metrics = {
    "total_documents": int(df_doc["doc_id"].nunique()) if "doc_id" in df_doc.columns else int(len(df_doc)),
    "total_requirements": int(len(df_req)),
    "avg_requirement_quality_score": round(float(safe_col(df_req, "requirement_quality_score").mean()), 2),
    "avg_document_quality_score": round(float(safe_col(df_doc, "document_quality_score").mean()), 2),
    "llm_rewrites_available": int(safe_col(df_req, "has_llm_rewrite", False).fillna(False).sum()),
    "top_problematic_count": int(len(df_top)),
}

print("Summary metrics:")
for k, v in summary_metrics.items():
    print(f"- {k}: {v}")

quality_distribution = (
    df_req["requirement_quality_level"].value_counts().reset_index()
    if "requirement_quality_level" in df_req.columns
    else pd.DataFrame()
)
if not quality_distribution.empty:
    quality_distribution.columns = ["requirement_quality_level", "count"]

action_distribution = (
    df_req["final_action_priority"].value_counts().reset_index()
    if "final_action_priority" in df_req.columns
    else pd.DataFrame()
)
if not action_distribution.empty:
    action_distribution.columns = ["final_action_priority", "count"]

document_status_distribution = (
    df_doc["final_document_status"].value_counts().reset_index()
    if "final_document_status" in df_doc.columns
    else pd.DataFrame()
)
if not document_status_distribution.empty:
    document_status_distribution.columns = ["final_document_status", "count"]

display(quality_distribution)
display(action_distribution)
display(document_status_distribution)

MODULE C12.1 — BUILD DASHBOARD ANALYTICS SUMMARY
Summary metrics:
- total_documents: 24
- total_requirements: 3608
- avg_requirement_quality_score: 86.28
- avg_document_quality_score: 84.45
- llm_rewrites_available: 100
- top_problematic_count: 3608


,requirement_quality_level,count
0,EXCELLENT,2293
1,GOOD,913
2,REVIEW_NEEDED,344
3,WEAK,57
4,CRITICAL,1


,final_action_priority,count
0,ACCEPTABLE,2493
1,SOURCE_PAGE_VISUAL_REVIEW,394
2,MANUAL_REVIEW,344
3,MODEL_UNCERTAINTY_REVIEW,319
4,REWRITE_REQUIRED,57
5,URGENT_REWRITE,1


,final_document_status,count
0,HIGH_QUALITY_DOCUMENT,14
1,GOOD_DOCUMENT,5
2,AMBIGUITY_REVIEW,4
3,CRITICAL_DOCUMENT_REVIEW,1


In [43]:
# ============================================================
# MODULE C12.2 — INSTALL AND IMPORT DASHBOARD LIBRARIES
# ============================================================

print("=" * 80)
print("MODULE C12.2 — DASHBOARD LIBRARIES")
print("=" * 80)

try:
    import gradio as gr
    import plotly.express as px
    import plotly.graph_objects as go
    print("Gradio and Plotly available.")
except ImportError:
    print("Installing missing dashboard libraries...")
    !pip install -q gradio plotly
    import gradio as gr
    import plotly.express as px
    import plotly.graph_objects as go
    print("Gradio and Plotly installed.")

MODULE C12.2 — DASHBOARD LIBRARIES
Gradio and Plotly available.


In [44]:
# ============================================================
# MODULE C12.3 — DASHBOARD HELPER FUNCTIONS
# ============================================================

print("=" * 80)
print("MODULE C12.3 — DASHBOARD HELPER FUNCTIONS")
print("=" * 80)

def make_overview_text():
    text = f"""
# Final Multimodal SRS Intelligence Dashboard

## Global summary

- Documents analyzed: **{summary_metrics['total_documents']}**
- Requirements analyzed: **{summary_metrics['total_requirements']}**
- Average requirement quality score: **{summary_metrics['avg_requirement_quality_score']}**
- Average document quality score: **{summary_metrics['avg_document_quality_score']}**
- LLM rewrites integrated: **{summary_metrics['llm_rewrites_available']}**
- Top problematic requirements available: **{summary_metrics['top_problematic_count']}**

## Pipeline components

1. Document structure extraction
2. Requirement extraction
3. Deep FR / NFR classification
4. NFR subtype classification
5. Ambiguity detection
6. Computer Vision page classification
7. Grad-CAM visual explanation
8. Multimodal requirement quality scoring
9. LLM-based requirement rewriting
10. Final integrated reporting
11. Visual analytics dashboard

## Interpretation

The dashboard combines NLP, Computer Vision, explainability, rule-based quality signals, deep learning quality prediction, and LLM rewriting into one final review interface.
"""
    return text


def make_requirement_quality_chart():
    if "requirement_quality_level" not in df_req.columns:
        return go.Figure()

    data = df_req["requirement_quality_level"].value_counts().reset_index()
    data.columns = ["quality_level", "count"]

    fig = px.bar(
        data,
        x="quality_level",
        y="count",
        title="Requirement Quality Level Distribution",
        text="count"
    )
    fig.update_layout(xaxis_title="Requirement quality level", yaxis_title="Count")
    return fig


def make_action_priority_chart():
    if "final_action_priority" not in df_req.columns:
        return go.Figure()

    data = df_req["final_action_priority"].value_counts().reset_index()
    data.columns = ["action_priority", "count"]

    fig = px.bar(
        data,
        x="action_priority",
        y="count",
        title="Final Action Priority Distribution",
        text="count"
    )
    fig.update_layout(xaxis_title="Final action priority", yaxis_title="Count")
    return fig


def make_document_status_chart():
    if "final_document_status" not in df_doc.columns:
        return go.Figure()

    data = df_doc["final_document_status"].value_counts().reset_index()
    data.columns = ["document_status", "count"]

    fig = px.pie(
        data,
        names="document_status",
        values="count",
        title="Document Status Distribution"
    )
    return fig


def make_score_scatter():
    needed = ["requirement_quality_score", "vision_confidence", "final_action_priority"]
    if not all(c in df_req.columns for c in needed):
        return go.Figure()

    plot_df = df_req.copy()
    plot_df["requirement_quality_score"] = pd.to_numeric(plot_df["requirement_quality_score"], errors="coerce")
    plot_df["vision_confidence"] = pd.to_numeric(plot_df["vision_confidence"], errors="coerce")
    plot_df = plot_df.dropna(subset=["requirement_quality_score", "vision_confidence"])

    fig = px.scatter(
        plot_df,
        x="vision_confidence",
        y="requirement_quality_score",
        color="final_action_priority",
        hover_data=[
            c for c in ["doc_id", "page_num", "requirement_quality_level", "final_ambiguity_label", "vision_page_type"]
            if c in plot_df.columns
        ],
        title="Requirement Quality vs Vision Confidence"
    )
    fig.update_layout(xaxis_title="Vision confidence", yaxis_title="Requirement quality score")
    return fig


def filter_requirements(doc_id, quality_level, action_priority, search_text, max_rows):
    data = df_req.copy()

    if doc_id and doc_id != "ALL" and "doc_id" in data.columns:
        data = data[data["doc_id"].astype(str) == str(doc_id)]

    if quality_level and quality_level != "ALL" and "requirement_quality_level" in data.columns:
        data = data[data["requirement_quality_level"].astype(str) == str(quality_level)]

    if action_priority and action_priority != "ALL" and "final_action_priority" in data.columns:
        data = data[data["final_action_priority"].astype(str) == str(action_priority)]

    if search_text and "block_text" in data.columns:
        data = data[data["block_text"].astype(str).str.contains(search_text, case=False, na=False)]

    keep_cols = [
        "doc_id", "page_num", "section_label_clean", "block_text",
        "requirement_quality_score", "requirement_quality_level",
        "final_prediction", "final_ambiguity_label",
        "vision_page_type", "vision_confidence",
        "final_action_priority",
        "quality_issues", "quality_recommendations",
        "has_llm_rewrite", "improved_requirement"
    ]

    keep_cols = [c for c in keep_cols if c in data.columns]
    data = data[keep_cols].head(int(max_rows))

    return data


def filter_documents(status, min_score):
    data = df_doc.copy()

    if status and status != "ALL" and "final_document_status" in data.columns:
        data = data[data["final_document_status"].astype(str) == str(status)]

    if "document_quality_score" in data.columns:
        data["document_quality_score"] = pd.to_numeric(data["document_quality_score"], errors="coerce")
        data = data[data["document_quality_score"] >= float(min_score)]

    keep_cols = [
        "doc_id", "total_requirements", "document_quality_score",
        "final_document_status", "document_review_priority",
        "ambiguous_requirements", "uncertain_ambiguity_requirements",
        "low_vision_confidence_requirements", "vision_page_mismatch_requirements",
        "llm_rewrites_available", "weak_or_critical_requirements",
        "ambiguous_ratio", "vision_mismatch_ratio",
        "document_vision_quality_flag"
    ]

    keep_cols = [c for c in keep_cols if c in data.columns]
    return data[keep_cols].sort_values(
        by="document_quality_score",
        ascending=True
    ) if "document_quality_score" in data.columns else data[keep_cols]


def get_problematic_requirements(max_rows):
    keep_cols = [
        "doc_id", "page_num", "block_text",
        "requirement_quality_score", "requirement_quality_level",
        "final_ambiguity_label", "vision_page_type", "vision_confidence",
        "final_action_priority", "quality_issues",
        "quality_recommendations", "improved_requirement"
    ]

    keep_cols = [c for c in keep_cols if c in df_top.columns]
    return df_top[keep_cols].head(int(max_rows))


def get_rewrites(max_rows):
    data = df_req.copy()

    if "has_llm_rewrite" in data.columns:
        data = data[data["has_llm_rewrite"] == True]
    elif "improved_requirement" in data.columns:
        data = data[data["improved_requirement"].notna()]

    keep_cols = [
        "doc_id", "page_num", "block_text", "improved_requirement",
        "requirement_quality_score", "requirement_quality_level",
        "rewrite_validation_score", "rewrite_validation_label",
        "rewrite_improvement_flag"
    ]

    keep_cols = [c for c in keep_cols if c in data.columns]
    return data[keep_cols].head(int(max_rows))


print("Dashboard helper functions ready.")

MODULE C12.3 — DASHBOARD HELPER FUNCTIONS
Dashboard helper functions ready.


In [45]:
# ============================================================
# MODULE C12.4 — LAUNCH FINAL GRADIO DASHBOARD
# ============================================================

print("=" * 80)
print("MODULE C12.4 — LAUNCH FINAL GRADIO DASHBOARD")
print("=" * 80)

doc_choices = ["ALL"] + sorted(df_req["doc_id"].astype(str).unique().tolist()) if "doc_id" in df_req.columns else ["ALL"]
quality_choices = ["ALL"] + sorted(df_req["requirement_quality_level"].dropna().astype(str).unique().tolist()) if "requirement_quality_level" in df_req.columns else ["ALL"]
action_choices = ["ALL"] + sorted(df_req["final_action_priority"].dropna().astype(str).unique().tolist()) if "final_action_priority" in df_req.columns else ["ALL"]
doc_status_choices = ["ALL"] + sorted(df_doc["final_document_status"].dropna().astype(str).unique().tolist()) if "final_document_status" in df_doc.columns else ["ALL"]

with gr.Blocks(title="Final Multimodal SRS Intelligence Dashboard") as dashboard:
    gr.Markdown("# Final Multimodal SRS Intelligence Dashboard")
    gr.Markdown(
        "Interactive visual analytics for requirement quality, ambiguity, computer vision context, LLM rewrites, and document-level reporting."
    )

    with gr.Tab("Overview"):
        gr.Markdown(make_overview_text())

        with gr.Row():
            gr.Plot(value=make_requirement_quality_chart())
            gr.Plot(value=make_action_priority_chart())

        with gr.Row():
            gr.Plot(value=make_document_status_chart())
            gr.Plot(value=make_score_scatter())

    with gr.Tab("Requirement Explorer"):
        with gr.Row():
            doc_filter = gr.Dropdown(doc_choices, value="ALL", label="Document")
            quality_filter = gr.Dropdown(quality_choices, value="ALL", label="Requirement quality level")
            action_filter = gr.Dropdown(action_choices, value="ALL", label="Action priority")

        with gr.Row():
            search_box = gr.Textbox(label="Search in requirement text", placeholder="Example: security, response time, user...")
            max_rows_req = gr.Slider(10, 500, value=50, step=10, label="Max rows")

        req_button = gr.Button("Filter requirements")
        req_table = gr.Dataframe(label="Filtered requirements", interactive=False)

        req_button.click(
            filter_requirements,
            inputs=[doc_filter, quality_filter, action_filter, search_box, max_rows_req],
            outputs=req_table
        )

    with gr.Tab("Document Report"):
        with gr.Row():
            status_filter = gr.Dropdown(doc_status_choices, value="ALL", label="Document status")
            min_score = gr.Slider(0, 100, value=0, step=1, label="Minimum document quality score")

        doc_button = gr.Button("Filter documents")
        doc_table = gr.Dataframe(label="Document-level final report", interactive=False)

        doc_button.click(
            filter_documents,
            inputs=[status_filter, min_score],
            outputs=doc_table
        )

    with gr.Tab("Top Problematic Requirements"):
        max_top = gr.Slider(10, 200, value=50, step=10, label="Max rows")
        top_button = gr.Button("Show top problematic requirements")
        top_table = gr.Dataframe(label="Top problematic requirements", interactive=False)

        top_button.click(
            get_problematic_requirements,
            inputs=[max_top],
            outputs=top_table
        )

    with gr.Tab("LLM Rewrites"):
        max_rewrites = gr.Slider(10, 200, value=50, step=10, label="Max rows")
        rewrite_button = gr.Button("Show LLM rewrites")
        rewrite_table = gr.Dataframe(label="Generated requirement rewrites", interactive=False)

        rewrite_button.click(
            get_rewrites,
            inputs=[max_rewrites],
            outputs=rewrite_table
        )

    with gr.Tab("Output Files"):
        gr.Markdown("## Final generated outputs")
        gr.Dataframe(value=df_paths, label="Generated report paths", interactive=False)

dashboard.launch(share=True, debug=False)

MODULE C12.4 — LAUNCH FINAL GRADIO DASHBOARD
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://51b082ecc342c6b632.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [46]:
# ============================================================
# MODULE C12.5 — SAVE DASHBOARD SUMMARY
# ============================================================

print("=" * 80)
print("MODULE C12.5 — SAVE DASHBOARD SUMMARY")
print("=" * 80)

c12_summary = f"""
C12 — Final Dashboard / Visual Analytics Summary

Purpose:
Provide an interactive visual analytics dashboard for the final multimodal SRS intelligence system.

Inputs:
- c11_final_requirement_report.csv
- c11_final_document_report.csv
- c11_top_problematic_requirements.csv
- c11_generated_report_paths.csv

Dashboard views:
1. Overview
   - Global project metrics
   - Requirement quality distribution
   - Final action priority distribution
   - Document status distribution
   - Requirement quality vs vision confidence scatter plot

2. Requirement Explorer
   - Filter by document
   - Filter by quality level
   - Filter by final action priority
   - Search by requirement text

3. Document Report
   - Review document-level quality status
   - Filter by document status
   - Filter by minimum document quality score

4. Top Problematic Requirements
   - Inspect requirements with highest risk
   - View quality issues, recommendations, and rewrite suggestions

5. LLM Rewrites
   - Review improved requirements generated by the instruction-tuned LLM
   - Compare original weak requirements with improved versions

6. Output Files
   - Lists final project outputs generated by C11

Dataset:
- Documents analyzed: {summary_metrics['total_documents']}
- Requirements analyzed: {summary_metrics['total_requirements']}
- Average requirement quality score: {summary_metrics['avg_requirement_quality_score']}
- Average document quality score: {summary_metrics['avg_document_quality_score']}
- LLM rewrites integrated: {summary_metrics['llm_rewrites_available']}

Conclusion:
C12 transforms the final multimodal SRS analysis outputs into an interactive review dashboard.
It helps users inspect requirement quality, ambiguity, visual context, document-level risks, and generated rewrites.
"""

c12_summary_path = "/kaggle/working/c12_dashboard_summary.txt"
with open(c12_summary_path, "w", encoding="utf-8") as f:
    f.write(c12_summary)

c12_manifest = pd.DataFrame([
    {"output_name": "dashboard_summary", "path": c12_summary_path},
    {"output_name": "source_requirement_report", "path": selected_c12_paths["final_requirement_report"]},
    {"output_name": "source_document_report", "path": selected_c12_paths["final_document_report"]},
    {"output_name": "source_top_problematic_requirements", "path": selected_c12_paths["top_problematic_requirements"]},
    {"output_name": "source_generated_report_paths", "path": selected_c12_paths["generated_report_paths"]},
])

c12_manifest_path = "/kaggle/working/c12_dashboard_outputs_manifest.csv"
c12_manifest.to_csv(c12_manifest_path, index=False)

print(c12_summary)
print("Saved:", c12_summary_path)
print("Saved:", c12_manifest_path)

display(c12_manifest)

MODULE C12.5 — SAVE DASHBOARD SUMMARY

C12 — Final Dashboard / Visual Analytics Summary

Purpose:
Provide an interactive visual analytics dashboard for the final multimodal SRS intelligence system.

Inputs:
- c11_final_requirement_report.csv
- c11_final_document_report.csv
- c11_top_problematic_requirements.csv
- c11_generated_report_paths.csv

Dashboard views:
1. Overview
   - Global project metrics
   - Requirement quality distribution
   - Final action priority distribution
   - Document status distribution
   - Requirement quality vs vision confidence scatter plot

2. Requirement Explorer
   - Filter by document
   - Filter by quality level
   - Filter by final action priority
   - Search by requirement text

3. Document Report
   - Review document-level quality status
   - Filter by document status
   - Filter by minimum document quality score

4. Top Problematic Requirements
   - Inspect requirements with highest risk
   - View quality issues, recommendations, and rewrite suggest

,output_name,path
0,dashboard_summary,/kaggle/working/c12_dashboard_summary.txt
1,source_requirement_report,/kaggle/working/c11_final_requirement_report.csv
2,source_document_report,/kaggle/working/c11_final_document_report.csv
3,source_top_problematic_requirements,/kaggle/working/c11_top_problematic_requiremen...
4,source_generated_report_paths,/kaggle/working/c11_generated_report_paths.csv


In [47]:
# ============================================================
# NOTEBOOK 4 — CREATE DEPLOYMENT BUNDLE FOR DJANGO
# C9 + C10 + C11 outputs
# ============================================================

from pathlib import Path
import shutil
import json
import pandas as pd
from IPython.display import FileLink, display

print("=" * 80)
print("CREATE NOTEBOOK 4 DEPLOYMENT BUNDLE")
print("=" * 80)

WORKING_DIR = Path("/kaggle/working")

bundle_dir = WORKING_DIR / "notebook4_deployment_bundle"
models_dir = bundle_dir / "models_ai"
configs_dir = bundle_dir / "configs"
outputs_dir = bundle_dir / "sample_outputs"
reports_dir = outputs_dir / "final_srs_reports"

# Clean old bundle
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)

models_dir.mkdir(parents=True, exist_ok=True)
configs_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Copy C9 trained quality model
# ------------------------------------------------------------

quality_model_src = WORKING_DIR / "roberta_requirement_quality_model"
quality_model_dst = models_dir / "roberta_requirement_quality_model"

if not quality_model_src.exists():
    raise FileNotFoundError(
        "Missing /kaggle/working/roberta_requirement_quality_model. "
        "Run C9 training cells first."
    )

shutil.copytree(quality_model_src, quality_model_dst)

print("Copied quality model:")
print(" ->", quality_model_dst)

# ------------------------------------------------------------
# 2. Save LLM rewrite config for C10
# ------------------------------------------------------------

llm_rewrite_config = {
    "module": "C10",
    "task": "requirement_rewriting",
    "llm_model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "model_type": "instruction_tuned_llm",
    "is_finetuned": False,
    "max_new_tokens": 256,
    "temperature": 0.2,
    "purpose": "Rewrite weak, ambiguous, or low-quality SRS requirements into clear, testable, measurable requirements.",
    "input_columns": [
        "block_text",
        "requirement_quality_score",
        "requirement_quality_level",
        "final_prediction",
        "final_ambiguity_label",
        "vision_page_type",
        "vision_confidence",
        "quality_issues",
        "quality_recommendations"
    ],
    "output_columns": [
        "improved_requirement",
        "rewrite_strategy",
        "what_was_fixed",
        "remaining_assumptions"
    ]
}

llm_config_path = configs_dir / "llm_rewrite_config.json"

with open(llm_config_path, "w", encoding="utf-8") as f:
    json.dump(llm_rewrite_config, f, indent=2)

print("Saved LLM rewrite config:")
print(" ->", llm_config_path)

# ------------------------------------------------------------
# 3. Save quality label config
# ------------------------------------------------------------

quality_label_config = {
    "module": "C9",
    "model_name": "roberta_requirement_quality_model",
    "model_family": "roberta-base",
    "task": "requirement_quality_classification",
    "labels": {
        "0": "LOW_QUALITY",
        "1": "MEDIUM_QUALITY",
        "2": "HIGH_QUALITY"
    },
    "input": "requirement text + multimodal/NLP quality signals",
    "output": "LOW_QUALITY / MEDIUM_QUALITY / HIGH_QUALITY"
}

quality_config_path = configs_dir / "quality_model_config.json"

with open(quality_config_path, "w", encoding="utf-8") as f:
    json.dump(quality_label_config, f, indent=2)

print("Saved quality model config:")
print(" ->", quality_config_path)

# ------------------------------------------------------------
# 4. Copy important C9, C10, C11 outputs
# ------------------------------------------------------------

important_files = [
    # C9 outputs
    "c9_multimodal_requirement_quality_scores.csv",
    "c9_document_quality_summary.csv",
    "c9_section_quality_summary.csv",
    "c9_multimodal_base_table.csv",
    "c9_roberta_quality_eval_results.csv",
    "c9_outputs_manifest.csv",
    "c9_final_summary.txt",

    # C10 outputs
    "c10_llm_requirement_rewrites.csv",
    "c10_llm_rewrite_summary.txt",

    # C11 outputs
    "c11_final_requirement_report.csv",
    "c11_final_document_report.csv",
    "c11_top_problematic_requirements.csv",
    "c11_generated_report_paths.csv",
    "c11_outputs_manifest.csv",
    "c11_final_project_summary.txt",

    # C12 optional outputs, if already generated
    "c12_dashboard_summary.txt",
    "c12_dashboard_outputs_manifest.csv",
]

copied_files = []
missing_files = []

for file_name in important_files:
    src = WORKING_DIR / file_name
    dst = outputs_dir / file_name

    if src.exists():
        shutil.copy2(src, dst)
        copied_files.append(file_name)
    else:
        missing_files.append(file_name)

print("\nCopied output files:")
for f in copied_files:
    print(" ->", f)

print("\nMissing optional files:")
for f in missing_files:
    print(" -", f)

# ------------------------------------------------------------
# 5. Copy final markdown reports folder
# ------------------------------------------------------------

final_reports_src = WORKING_DIR / "final_srs_reports"
final_reports_dst = reports_dir

if final_reports_src.exists():
    shutil.copytree(final_reports_src, final_reports_dst)
    print("\nCopied final_srs_reports folder:")
    print(" ->", final_reports_dst)
else:
    print("\nWarning: final_srs_reports folder not found.")

# ------------------------------------------------------------
# 6. Create deployment manifest
# ------------------------------------------------------------

deployment_manifest = {
    "bundle_name": "notebook4_deployment_bundle",
    "purpose": "C9-C10-C11 deployment assets for Django SRS intelligence system",
    "contains": {
        "quality_model": str(quality_model_dst),
        "configs": [
            str(llm_config_path),
            str(quality_config_path)
        ],
        "outputs": copied_files,
        "reports_folder": str(final_reports_dst) if final_reports_dst.exists() else None
    },
    "django_usage": {
        "C9": "Load roberta_requirement_quality_model to classify requirement quality.",
        "C10": "Use Qwen/Qwen2.5-1.5B-Instruct config for requirement rewriting.",
        "C11": "Use final CSV/Markdown outputs as report templates and test references."
    }
}

manifest_path = configs_dir / "deployment_manifest_notebook4.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(deployment_manifest, f, indent=2)

print("\nSaved deployment manifest:")
print(" ->", manifest_path)

# ------------------------------------------------------------
# 7. Zip the full bundle
# ------------------------------------------------------------

zip_base = WORKING_DIR / "notebook4_deployment_bundle"
zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(bundle_dir)
)

print("\n" + "=" * 80)
print("NOTEBOOK 4 DEPLOYMENT BUNDLE READY")
print("=" * 80)
print("Bundle folder:", bundle_dir)
print("Zip file:", zip_path)

display(FileLink(zip_path))

CREATE NOTEBOOK 4 DEPLOYMENT BUNDLE
Copied quality model:
 -> /kaggle/working/notebook4_deployment_bundle/models_ai/roberta_requirement_quality_model
Saved LLM rewrite config:
 -> /kaggle/working/notebook4_deployment_bundle/configs/llm_rewrite_config.json
Saved quality model config:
 -> /kaggle/working/notebook4_deployment_bundle/configs/quality_model_config.json

Copied output files:
 -> c9_multimodal_requirement_quality_scores.csv
 -> c9_document_quality_summary.csv
 -> c9_section_quality_summary.csv
 -> c9_multimodal_base_table.csv
 -> c9_roberta_quality_eval_results.csv
 -> c9_outputs_manifest.csv
 -> c9_final_summary.txt
 -> c10_llm_requirement_rewrites.csv
 -> c10_llm_rewrite_summary.txt
 -> c11_final_requirement_report.csv
 -> c11_final_document_report.csv
 -> c11_top_problematic_requirements.csv
 -> c11_generated_report_paths.csv
 -> c11_outputs_manifest.csv
 -> c11_final_project_summary.txt
 -> c12_dashboard_summary.txt
 -> c12_dashboard_outputs_manifest.csv

Missing optional 

/kaggle/working/notebook4_deployment_bundle.zip

In [48]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

bundle_dir = Path("/kaggle/working/notebook4_deployment_bundle")
new_zip_base = "/kaggle/working/notebook4_deployment_bundle_v2"

if not bundle_dir.exists():
    raise FileNotFoundError("Bundle folder not found.")

new_zip_path = shutil.make_archive(
    base_name=new_zip_base,
    format="zip",
    root_dir=str(bundle_dir)
)

print("Created:", new_zip_path)
print("Size MB:", round(Path(new_zip_path).stat().st_size / (1024 * 1024), 2))

display(FileLink(new_zip_path))

Created: /kaggle/working/notebook4_deployment_bundle_v2.zip
Size MB: 406.7


/kaggle/working/notebook4_deployment_bundle_v2.zip

In [49]:
from pathlib import Path

for p in [
    "/kaggle/working/notebook4_deployment_bundle.zip",
    "/kaggle/working/notebook4_deployment_bundle_v2.zip"
]:
    p = Path(p)
    print(p.name)
    print("exists:", p.exists())
    if p.exists():
        print("size MB:", round(p.stat().st_size / (1024 * 1024), 2))
        print("size GB:", round(p.stat().st_size / (1024 * 1024 * 1024), 2))
    print("-" * 50)

notebook4_deployment_bundle.zip
exists: True
size MB: 406.7
size GB: 0.4
--------------------------------------------------
notebook4_deployment_bundle_v2.zip
exists: True
size MB: 406.7
size GB: 0.4
--------------------------------------------------


In [50]:
from pathlib import Path
import shutil
import json
import os

print("=" * 80)
print("CREATE LIGHTWEIGHT NOTEBOOK 4 DEPLOYMENT BUNDLE")
print("=" * 80)

bundle_dir = Path("/kaggle/working/notebook4_deployment_bundle_light")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)

models_dir = bundle_dir / "models"
outputs_dir = bundle_dir / "outputs"
configs_dir = bundle_dir / "configs"
reports_dir = bundle_dir / "sample_reports"

models_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)
configs_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# 1. Copy final RoBERTa quality model only
quality_model_src = Path("/kaggle/working/roberta_requirement_quality_model")
quality_model_dst = models_dir / "roberta_requirement_quality_model"

if quality_model_src.exists():
    shutil.copytree(quality_model_src, quality_model_dst)
    print("Copied quality model:", quality_model_dst)
else:
    print("WARNING: Missing quality model:", quality_model_src)

# 2. Save LLM rewrite config
llm_config = {
    "llm_model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "task": "requirement_rewriting",
    "max_new_tokens": 256,
    "temperature": 0.2,
    "do_sample": False
}

with open(configs_dir / "llm_rewrite_config.json", "w", encoding="utf-8") as f:
    json.dump(llm_config, f, indent=2)

# 3. Save quality model config
quality_config = {
    "model_type": "roberta-base",
    "task": "requirement_quality_classification",
    "labels": ["LOW_QUALITY", "MEDIUM_QUALITY", "HIGH_QUALITY"],
    "input": "requirement text + multimodal quality signals",
    "output": "deep_quality_label + confidence"
}

with open(configs_dir / "quality_model_config.json", "w", encoding="utf-8") as f:
    json.dump(quality_config, f, indent=2)

# 4. Copy important final outputs only
important_files = [
    "/kaggle/working/c9_multimodal_requirement_quality_scores.csv",
    "/kaggle/working/c9_document_quality_summary.csv",
    "/kaggle/working/c9_section_quality_summary.csv",
    "/kaggle/working/c10_llm_requirement_rewrites.csv",
    "/kaggle/working/c10_llm_rewrite_summary.txt",
    "/kaggle/working/c11_final_requirement_report.csv",
    "/kaggle/working/c11_final_document_report.csv",
    "/kaggle/working/c11_top_problematic_requirements.csv",
    "/kaggle/working/c11_generated_report_paths.csv",
    "/kaggle/working/c11_final_project_summary.txt",
    "/kaggle/working/c12_dashboard_summary.txt",
    "/kaggle/working/c12_dashboard_outputs_manifest.csv",
]

for file_path in important_files:
    src = Path(file_path)
    if src.exists():
        shutil.copy2(src, outputs_dir / src.name)
        print("Copied:", src.name)
    else:
        print("Missing optional:", src.name)

# 5. Copy only some markdown reports, not the full folder if huge
reports_src = Path("/kaggle/working/final_srs_reports")
if reports_src.exists():
    md_files = sorted(list(reports_src.glob("*.md")))[:5]
    for md in md_files:
        shutil.copy2(md, reports_dir / md.name)
    print(f"Copied {len(md_files)} sample markdown reports.")
else:
    print("No final_srs_reports folder found.")

# 6. Create zip
zip_base = "/kaggle/working/notebook4_deployment_bundle_light"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(bundle_dir))

print("=" * 80)
print("LIGHT BUNDLE READY")
print("Zip path:", zip_path)
print("Size MB:", round(Path(zip_path).stat().st_size / (1024 * 1024), 2))
print("=" * 80)

CREATE LIGHTWEIGHT NOTEBOOK 4 DEPLOYMENT BUNDLE
Copied quality model: /kaggle/working/notebook4_deployment_bundle_light/models/roberta_requirement_quality_model
Copied: c9_multimodal_requirement_quality_scores.csv
Copied: c9_document_quality_summary.csv
Copied: c9_section_quality_summary.csv
Copied: c10_llm_requirement_rewrites.csv
Copied: c10_llm_rewrite_summary.txt
Copied: c11_final_requirement_report.csv
Copied: c11_final_document_report.csv
Copied: c11_top_problematic_requirements.csv
Copied: c11_generated_report_paths.csv
Copied: c11_final_project_summary.txt
Copied: c12_dashboard_summary.txt
Copied: c12_dashboard_outputs_manifest.csv
Copied 5 sample markdown reports.
LIGHT BUNDLE READY
Zip path: /kaggle/working/notebook4_deployment_bundle_light.zip
Size MB: 406.42
